In [ ]:
from kbpaths import KB_DATA, KB_PROJECT, KB_RESULTS, KB_XENIUM_RAW  # repository path config (config/kbpaths.py)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import random
import warnings
from datetime import datetime

import gdown
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns
import squidpy as sq
from matplotlib import gridspec
from sklearn.preprocessing import MinMaxScaler

from nichecompass.models import NicheCompass
from nichecompass.utils import (add_gps_from_gp_dict_to_adata,
                                compute_communication_gp_network,
                                visualize_communication_gp_network,
                                create_new_color_dict,
                                extract_gp_dict_from_mebocost_ms_interactions,
                                extract_gp_dict_from_nichenet_lrt_interactions,
                                extract_gp_dict_from_omnipath_lr_interactions,
                                filter_and_combine_gp_dict_gps_v2,
                                generate_enriched_gp_info_plots)

In [ ]:
### Dataset ###
dataset = "kidney_biopsy"
species = "human"
spatial_key = "spatial"
n_neighbors = 4

### Model ###
# AnnData Keys
counts_key = "counts"
adj_key = "spatial_connectivities"
gp_names_key = "nichecompass_gp_names"
active_gp_names_key = "nichecompass_active_gp_names"
gp_targets_mask_key = "nichecompass_gp_targets"
gp_targets_categories_mask_key = "nichecompass_gp_targets_categories"
gp_sources_mask_key = "nichecompass_gp_sources"
gp_sources_categories_mask_key = "nichecompass_gp_sources_categories"
latent_key = "nichecompass_latent"

# Architecture
conv_layer_encoder = "gatv2conv" # change to "gatv2conv" if enough compute and memory
active_gp_thresh_ratio = 0.01

# Trainer
n_epochs = 400
n_epochs_all_gps = 25
lr = 0.001
lambda_edge_recon = 500000.
lambda_gene_expr_recon = 300.
lambda_l1_masked = 0. # prior GP  regularization
lambda_l1_addon = 30. # de novo GP regularization
edge_batch_size = 1024 # increase if more memory available or decrease to save memory
n_sampled_neighbors = 4
use_cuda_if_available = True

### Analysis ###
cell_type_key = "cell_type_manual"
latent_leiden_resolution = 0.4
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"
sample_key = "TMA_ID"
spot_size = 15
differential_gp_test_results_key = "nichecompass_differential_gp_test_results"

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
# Get time of notebook execution for timestamping saved artifacts
now = datetime.now()
current_timestamp = now.strftime("%d%m%Y_%H%M%S")

In [ ]:
current_timestamp = "02092025_162433"

In [ ]:
# Define paths
ga_data_folder_path = f"{KB_PROJECT}/niche_compass_results/data/gene_annotations"
gp_data_folder_path = f"{KB_PROJECT}/niche_compass_results/data/gene_programs"
so_data_folder_path = f"{KB_PROJECT}/niche_compass_resultsdata/spatial_omics"
omnipath_lr_network_file_path = f"{gp_data_folder_path}/omnipath_lr_network.csv"
collectri_tf_network_file_path = f"{gp_data_folder_path}/collectri_tf_network_{species}.csv"
nichenet_lr_network_file_path = f"{gp_data_folder_path}/nichenet_lr_network_v2_{species}.csv"
nichenet_ligand_target_matrix_file_path = f"{gp_data_folder_path}/nichenet_ligand_target_matrix_v2_{species}.csv"
mebocost_enzyme_sensor_interactions_folder_path = f"{gp_data_folder_path}/metabolite_enzyme_sensor_gps"
gene_orthologs_mapping_file_path = f"{ga_data_folder_path}/human_mouse_gene_orthologs.csv"
artifacts_folder_path = f"{KB_PROJECT}/niche_compass_results/artifacts"
model_folder_path = f"{artifacts_folder_path}/single_sample/{current_timestamp}/model"
figure_folder_path = f"{artifacts_folder_path}/single_sample/{current_timestamp}/figures"

In [ ]:
os.makedirs(model_folder_path, exist_ok=True)
os.makedirs(figure_folder_path, exist_ok=True)
os.makedirs(so_data_folder_path, exist_ok=True)

In [ ]:
os.makedirs(ga_data_folder_path, exist_ok=True)
os.makedirs(gp_data_folder_path, exist_ok=True)
os.makedirs(so_data_folder_path, exist_ok=True)

In [ ]:
import os
os.makedirs(f"{KB_PROJECT}/niche_compass_results/data/gene_annotations", exist_ok=True)

In [ ]:
import pandas as pd

# Load the OmniPath file
omnipath_df = pd.read_csv(omnipath_lr_network_file_path)
print(f"Original shape: {omnipath_df.shape}")
print(f"Missing values in genesymbol_intercell_target: {omnipath_df['genesymbol_intercell_target'].isna().sum()}")

# Remove rows with missing target gene symbols
omnipath_df_cleaned = omnipath_df.dropna(subset=['genesymbol_intercell_target'])
print(f"Cleaned shape: {omnipath_df_cleaned.shape}")

# Also check and clean source column if needed
print(f"Missing values in genesymbol_intercell_source: {omnipath_df_cleaned['genesymbol_intercell_source'].isna().sum()}")
omnipath_df_cleaned = omnipath_df_cleaned.dropna(subset=['genesymbol_intercell_source'])
print(f"Final cleaned shape: {omnipath_df_cleaned.shape}")

# Save the cleaned file
omnipath_df_cleaned.to_csv(omnipath_lr_network_file_path, index=False)
print("Saved cleaned OmniPath file")

# Now try the function again
omnipath_gp_dict = extract_gp_dict_from_omnipath_lr_interactions(
    species="human",
    load_from_disk=True,
    save_to_disk=True,
    lr_network_file_path=omnipath_lr_network_file_path,
    gene_orthologs_mapping_file_path=None,
    plot_gp_gene_count_distributions=True,
    gp_gene_count_distributions_save_path=f"{figure_folder_path}/omnipath_gp_gene_count_distributions.pdf"
)

In [ ]:
# Display example OmniPath GP
omnipath_gp_names = list(omnipath_gp_dict.keys())
random.shuffle(omnipath_gp_names)
omnipath_gp_name = omnipath_gp_names[0]
print(f"{omnipath_gp_name}: {omnipath_gp_dict[omnipath_gp_name]}")

In [ ]:
# Retrieve NicheNet GPs (source: ligand genes; target: receptor genes, target genes)
nichenet_gp_dict = extract_gp_dict_from_nichenet_lrt_interactions(
    species=species,
    version="v2",
    keep_target_genes_ratio=1.,
    max_n_target_genes_per_gp=250,
    load_from_disk=False,
    save_to_disk=True,
    lr_network_file_path=nichenet_lr_network_file_path,
    ligand_target_matrix_file_path=nichenet_ligand_target_matrix_file_path,
    gene_orthologs_mapping_file_path=gene_orthologs_mapping_file_path,
    plot_gp_gene_count_distributions=True)

In [ ]:
# Display example NicheNet GP
nichenet_gp_names = list(nichenet_gp_dict.keys())
random.shuffle(nichenet_gp_names)
nichenet_gp_name = nichenet_gp_names[0]
print(f"{nichenet_gp_name}: {nichenet_gp_dict[nichenet_gp_name]}")

In [ ]:
import os

# Check if the directory exists and what's in it
print(f"MEBOCOST directory exists: {os.path.exists(mebocost_enzyme_sensor_interactions_folder_path)}")
if os.path.exists(mebocost_enzyme_sensor_interactions_folder_path):
    print(f"Contents: {os.listdir(mebocost_enzyme_sensor_interactions_folder_path)}")
else:
    print("Creating MEBOCOST directory...")
    os.makedirs(mebocost_enzyme_sensor_interactions_folder_path, exist_ok=True)

In [ ]:
# Retrieve MEBOCOST GPs (source: enzyme genes; target: sensor genes)
mebocost_gp_dict = extract_gp_dict_from_mebocost_ms_interactions(
    dir_path=mebocost_enzyme_sensor_interactions_folder_path,
    species=species,
    plot_gp_gene_count_distributions=True)

In [ ]:
# Display example MEBOCOST GP
mebocost_gp_names = list(mebocost_gp_dict.keys())
random.shuffle(mebocost_gp_names)
mebocost_gp_name = mebocost_gp_names[0]
print(f"{mebocost_gp_name}: {mebocost_gp_dict[mebocost_gp_name]}")

In [ ]:
# Filter and combine GPs
gp_dicts = [omnipath_gp_dict, nichenet_gp_dict, mebocost_gp_dict]
combined_gp_dict = filter_and_combine_gp_dict_gps_v2(
    gp_dicts,
    verbose=True)

print(f"Number of gene programs after filtering and combining: "
      f"{len(combined_gp_dict)}.")

In [ ]:
adata = sc.read_h5ad(f"{KB_XENIUM_RAW}/h5ad_files_only_transcript/marker_genes/clustered_annotated_final_file.h5ad")

In [ ]:
import numpy as np
import pandas as pd

# Build a set of labels to exclude (spellings must match your obs exactly)
# Cores excluded: unassigned cells, the Apollo d0 A2 control core and nine Utrecht cores (K4 DN, K24 TIN, K29 MN,
# K23 TIN, K25 TIN, K71 FSGS, K46 MN, K78 FSGS, K60 IgAN). Internal labels contain pathology numbers -> private
# list (config/cores.py).
from cores import excluded_core_labels
to_exclude = set(excluded_core_labels("notebook2025_blacklist"))

# Create a boolean mask: keep rows not in to_exclude and not NaN
mask_keep = ~adata.obs["TMA_ID"].isin(to_exclude) & adata.obs["TMA_ID"].notna()

# Subset AnnData (this returns a view; use .copy() if you want an independent object)
adata_filt = adata[mask_keep].copy()

# Optional: drop unused categories if TMA_ID is categorical
if pd.api.types.is_categorical_dtype(adata_filt.obs["TMA_ID"]):
    adata_filt.obs["TMA_ID"] = adata_filt.obs["TMA_ID"].cat.remove_unused_categories()

adata_filt

adata = adata_filt.copy()

In [ ]:
# Compute spatial neighborhood
sq.gr.spatial_neighbors(adata,
                        coord_type="generic",
                        spatial_key=spatial_key,
                        n_neighs=n_neighbors)

# Make adjacency matrix symmetric
adata.obsp[adj_key] = (
    adata.obsp[adj_key].maximum(
        adata.obsp[adj_key].T))

In [ ]:
# Add the GP dictionary as binary masks to the adata
add_gps_from_gp_dict_to_adata(
    gp_dict=combined_gp_dict,
    adata=adata,
    gp_targets_mask_key=gp_targets_mask_key,
    gp_targets_categories_mask_key=gp_targets_categories_mask_key,
    gp_sources_mask_key=gp_sources_mask_key,
    gp_sources_categories_mask_key=gp_sources_categories_mask_key,
    gp_names_key=gp_names_key,
    min_genes_per_gp=2,
    min_source_genes_per_gp=1,
    min_target_genes_per_gp=1,
    max_genes_per_gp=None,
    max_source_genes_per_gp=None,
    max_target_genes_per_gp=None)

In [ ]:
cell_type_colors = create_new_color_dict(
    adata=adata,
    cat_key=cell_type_key)

In [ ]:
print(f"Number of nodes (observations): {adata.layers['counts'].shape[0]}")
print(f"Number of node features (genes): {adata.layers['counts'].shape[1]}")

# Visualize cell-level annotated data in physical space
sc.pl.spatial(adata,
              color=cell_type_key,
              palette=cell_type_colors,
              spot_size=spot_size)        

In [ ]:
# Initialize model
model = NicheCompass(adata,
                     counts_key=counts_key,
                     adj_key=adj_key,
                     gp_names_key=gp_names_key,
                     active_gp_names_key=active_gp_names_key,
                     gp_targets_mask_key=gp_targets_mask_key,
                     gp_targets_categories_mask_key=gp_targets_categories_mask_key,
                     gp_sources_mask_key=gp_sources_mask_key,
                     gp_sources_categories_mask_key=gp_sources_categories_mask_key,
                     latent_key=latent_key,
                     conv_layer_encoder=conv_layer_encoder,
                     active_gp_thresh_ratio=active_gp_thresh_ratio)

In [ ]:
# Train model
model.train(n_epochs=n_epochs,
            n_epochs_all_gps=n_epochs_all_gps,
            lr=lr,
            lambda_edge_recon=lambda_edge_recon,
            lambda_gene_expr_recon=lambda_gene_expr_recon,
            lambda_l1_masked=lambda_l1_masked,
            edge_batch_size=edge_batch_size,
            n_sampled_neighbors=n_sampled_neighbors,
            use_cuda_if_available=use_cuda_if_available,
            verbose=False)

In [ ]:
# Compute latent neighbor graph
sc.pp.neighbors(model.adata,
                use_rep=latent_key,
                key_added=latent_key)

# Compute UMAP embedding
sc.tl.umap(model.adata,
           neighbors_key=latent_key)

In [ ]:
# Save trained model
model.save(dir_path=model_folder_path,
           overwrite=True,
           save_adata=True,
           adata_file_name="adata.h5ad")

In [ ]:
load_timestamp = "02092025_162433"
#load_timestamp = current_timestamp # uncomment if you trained the model in this notebook

figure_folder_path = f"{artifacts_folder_path}/single_sample/{load_timestamp}/figures"
model_folder_path = f"{artifacts_folder_path}/single_sample/{load_timestamp}/model"

os.makedirs(figure_folder_path, exist_ok=True)

In [ ]:
# Load trained model
model = NicheCompass.load(dir_path=model_folder_path,
                          adata=None,
                          adata_file_name="adata.h5ad",
                          gp_names_key=gp_names_key)

In [ ]:
samples = model.adata.obs[sample_key].unique().tolist()

In [ ]:
cell_type_colors = create_new_color_dict(
    adata=model.adata,
    cat_key=cell_type_key)

In [ ]:
# Create plot of cell type annotations in physical and latent space
groups = None
save_fig = True
file_path = f"{figure_folder_path}/" \
            "cell_types_latent_physical_space.pdf"

fig = plt.figure(figsize=(12, 14))
title = fig.suptitle(t=f"Cell Types " \
                       "in Latent and Physical Space",
                     y=0.96,
                     x=0.55,
                     fontsize=20)
spec1 = gridspec.GridSpec(ncols=1,
                          nrows=2,
                          width_ratios=[1],
                          height_ratios=[3, 2])
spec2 = gridspec.GridSpec(ncols=len(samples),
                          nrows=2,
                          width_ratios=[1] * len(samples),
                          height_ratios=[3, 2])
axs = []
axs.append(fig.add_subplot(spec1[0]))
sc.pl.umap(adata=model.adata,
           color=[cell_type_key],
           groups=groups,palette=cell_type_colors,
           title=f"Cell Types in Latent Space",
           ax=axs[0],
           show=False)
for idx, sample in enumerate(samples):
    axs.append(fig.add_subplot(spec2[len(samples) + idx]))
    sc.pl.spatial(adata=model.adata[model.adata.obs[sample_key] == sample],
                  color=[cell_type_key],
                  groups=groups,
                  palette=cell_type_colors,
                  spot_size=spot_size,
                  title=f"Cell Types in Physical Space \n"
                        f"(Sample: {sample})",
                  legend_loc=None,
                  ax=axs[idx+1],
                  show=False)

# Create and position shared legend
handles, labels = axs[0].get_legend_handles_labels()
lgd = fig.legend(handles,
                 labels,
                 loc="center left",
                 bbox_to_anchor=(0.98, 0.5))
axs[0].get_legend().remove()

# Adjust, save and display plot
plt.subplots_adjust(wspace=0.2, hspace=0.25)
if save_fig:
    fig.savefig(file_path,
                bbox_extra_artists=(lgd, title),
                bbox_inches="tight")
plt.show()

In [ ]:
# Compute latent Leiden clustering
sc.tl.leiden(adata=model.adata,
             resolution=latent_leiden_resolution,
             key_added=latent_cluster_key,
             neighbors_key=latent_key)

In [ ]:
latent_cluster_colors = create_new_color_dict(
    adata=model.adata,
    cat_key=latent_cluster_key)

In [ ]:
# Create plot of latent cluster / niche annotations in physical and latent space
groups = None # set this to a specific cluster for easy visualization, e.g. ["17"]
save_fig = True
file_path = f"{figure_folder_path}/" \
            f"res_{latent_leiden_resolution}_" \
            "niches_latent_physical_space.pdf"

fig = plt.figure(figsize=(12, 14))
title = fig.suptitle(t=f"NicheCompass Niches " \
                       "in Latent and Physical Space",
                     y=0.96,
                     x=0.55,
                     fontsize=20)
spec1 = gridspec.GridSpec(ncols=1,
                          nrows=2,
                          width_ratios=[1],
                          height_ratios=[3, 2])
spec2 = gridspec.GridSpec(ncols=len(samples),
                          nrows=2,
                          width_ratios=[1] * len(samples),
                          height_ratios=[3, 2])
axs = []
axs.append(fig.add_subplot(spec1[0]))
sc.pl.umap(adata=model.adata,
           color=[latent_cluster_key],
           groups=groups,
           palette=latent_cluster_colors,
           title=f"Niches in Latent Space",
           ax=axs[0],
           show=False)
for idx, sample in enumerate(samples):
    axs.append(fig.add_subplot(spec2[len(samples) + idx]))
    sc.pl.spatial(adata=model.adata[model.adata.obs[sample_key] == sample],
                  color=[latent_cluster_key],
                  groups=groups,
                  palette=latent_cluster_colors,
                  spot_size=spot_size,
                  title=f"Niches in Physical Space \n"
                        f"(Sample: {sample})",
                  legend_loc=None,
                  ax=axs[idx+1],
                  show=False)

# Create and position shared legend
handles, labels = axs[0].get_legend_handles_labels()
lgd = fig.legend(handles,
                 labels,
                 loc="center left",
                 bbox_to_anchor=(0.98, 0.5))
axs[0].get_legend().remove()

# Adjust, save and display plot
plt.subplots_adjust(wspace=0.2, hspace=0.25)
if save_fig:
    fig.savefig(file_path,
                bbox_extra_artists=(lgd, title),
                bbox_inches="tight")
plt.show()

In [ ]:
save_fig = True
file_path = f"{figure_folder_path}/" \
            f"res_{latent_leiden_resolution}_" \
            f"niche_composition_cell_type.pdf"

df_counts = (model.adata.obs.groupby([latent_cluster_key, cell_type_key])
             .size().unstack())
df_counts.plot(kind="bar", stacked=True, figsize=(10,10))
legend = plt.legend(bbox_to_anchor=(1, 1), loc="upper left", prop={'size': 10})
legend.set_title("Cell Type Annotations", prop={'size': 10})
plt.title("Cell Type Composition of Niches")
plt.xlabel("Niche")
plt.ylabel("Cell Counts")
if save_fig:
    plt.savefig(file_path,
                bbox_extra_artists=(legend,),
                bbox_inches="tight")

In [ ]:
save_fig = True
file_path = f"{figure_folder_path}/" \
            f"res_{latent_leiden_resolution}_" \
            f"niche_composition_condition.pdf"

df_counts = (model.adata.obs.groupby([latent_cluster_key, "condition"])
             .size().unstack())
df_counts.plot(kind="bar", stacked=True, figsize=(10,10))
legend = plt.legend(bbox_to_anchor=(1, 1), loc="upper left", prop={'size': 10})
legend.set_title("Condition Annotations", prop={'size': 10})
plt.title("Condition Composition of Niches")
plt.xlabel("Niche")
plt.ylabel("Cell Counts")
if save_fig:
    plt.savefig(file_path,
                bbox_extra_artists=(legend,),
                bbox_inches="tight")

In [ ]:
save_fig = True
file_path = f"{figure_folder_path}/" \
            f"res_{latent_leiden_resolution}_" \
            f"niche_composition_sample.pdf"

df_counts = (model.adata.obs.groupby([latent_cluster_key, "TMA_ID"])
             .size().unstack())
df_counts.plot(kind="bar", stacked=True, figsize=(10,10))
legend = plt.legend(bbox_to_anchor=(1, 1), loc="upper left", prop={'size': 10})
legend.set_title("Condition Annotations", prop={'size': 10})
plt.title("Sample Composition of Niches")
plt.xlabel("Niche")
plt.ylabel("Cell Counts")
if save_fig:
    plt.savefig(file_path,
                bbox_extra_artists=(legend,),
                bbox_inches="tight")

In [ ]:
# Check number of active GPs
active_gps = model.get_active_gps()
print(f"Number of total gene programs: {len(model.adata.uns[gp_names_key])}.")
print(f"Number of active gene programs: {len(active_gps)}.")

In [ ]:
# Display example active GPs
gp_summary_df = model.get_gp_summary()
gp_summary_df[gp_summary_df["gp_active"] == True].head()

In [ ]:
# Set parameters for differential gp testing
selected_cats = None
comparison_cats = "rest"
title = f"NicheCompass Strongly Enriched Niche GPs"
log_bayes_factor_thresh = 2.3
save_fig = True
file_path = f"{figure_folder_path}/" \
            f"/log_bayes_factor_{log_bayes_factor_thresh}" \
             "_niches_enriched_gps_heatmap.pdf"

In [ ]:
# Run differential gp testing
enriched_gps = model.run_differential_gp_tests(
    cat_key=latent_cluster_key,
    selected_cats=selected_cats,
    comparison_cats=comparison_cats,
    log_bayes_factor_thresh=log_bayes_factor_thresh)

In [ ]:
# Results are stored in a df in the adata object
model.adata.uns[differential_gp_test_results_key]

In [ ]:
# Visualize GP activities of enriched GPs across niches
df = model.adata.obs[[latent_cluster_key] + enriched_gps].groupby(latent_cluster_key).mean()

scaler = MinMaxScaler()
normalized_columns = scaler.fit_transform(df)
normalized_df = pd.DataFrame(normalized_columns, columns=df.columns)
normalized_df.index = df.index

plt.figure(figsize=(16, 8))  # Set the figure size
ax = sns.heatmap(normalized_df,
            cmap='viridis',
            annot=False,
            linewidths=0)
plt.xticks(rotation=45,
           fontsize=8,
           ha="right"
          )
plt.xlabel("Gene Programs", fontsize=16)
plt.savefig(f"{figure_folder_path}/enriched_gps_heatmap.pdf",
            bbox_inches="tight")

In [ ]:
# Store gene program summary of enriched gene programs
save_file = True
file_path = f"{figure_folder_path}/" \
            f"/log_bayes_factor_{log_bayes_factor_thresh}_" \
            "niche_enriched_gps_summary.csv"

gp_summary_cols = ["gp_name",
                   "n_source_genes",
                   "n_non_zero_source_genes",
                   "n_target_genes",
                   "n_non_zero_target_genes",
                   "gp_source_genes",
                   "gp_target_genes",
                   "gp_source_genes_importances",
                   "gp_target_genes_importances"]

enriched_gp_summary_df = gp_summary_df[gp_summary_df["gp_name"].isin(enriched_gps)]
cat_dtype = pd.CategoricalDtype(categories=enriched_gps, ordered=True)
enriched_gp_summary_df.loc[:, "gp_name"] = enriched_gp_summary_df["gp_name"].astype(cat_dtype)
enriched_gp_summary_df = enriched_gp_summary_df.sort_values(by="gp_name")
enriched_gp_summary_df = enriched_gp_summary_df[gp_summary_cols]

if save_file:
    enriched_gp_summary_df.to_csv(f"{file_path}")
else:
    display(enriched_gp_summary_df)

In [ ]:
"""plot_label = f"log_bayes_factor_{log_bayes_factor_thresh}_cluster_{selected_cats[0] if selected_cats else 'None'}_vs_rest"
save_figs = True

generate_enriched_gp_info_plots(
    plot_label=plot_label,
    model=model,
    sample_key=sample_key,
    differential_gp_test_results_key=differential_gp_test_results_key,
    cat_key=latent_cluster_key,
    cat_palette=latent_cluster_colors,
    n_top_enriched_gp_start_idx=0,
    n_top_enriched_gp_end_idx=10,
    feature_spaces=samples, # ["latent"]
    n_top_genes_per_gp=3,
    save_figs=save_figs,
    figure_folder_path=f"{figure_folder_path}/",
    spot_size=spot_size)"""

In [ ]:
gp_name = "THBS1_combined_GP"

In [ ]:
network_df = compute_communication_gp_network(
    gp_list=[gp_name],
    model=model,
    group_key=latent_cluster_key,
    n_neighbors=n_neighbors)

visualize_communication_gp_network(
    adata=model.adata,
    network_df=network_df,
    figsize=(10, 7),
    cat_colors=latent_cluster_colors,
    edge_type_colors=["#1f77b4"], 
    cat_key=latent_cluster_key,
    save=True,
    save_path=f"{figure_folder_path}/gp_network_{gp_name}.pdf",
    )

In [ ]:
model.adata

In [ ]:
import pandas as pd
import numpy as np

# 1.1 GP summary with importances (already shown briefly in the tutorial)
gp_summary_df = model.get_gp_summary()
# Columns include:
# ['gp_name','all_gp_idx','gp_active','active_gp_idx',
#  'n_source_genes','n_non_zero_source_genes','n_target_genes','n_non_zero_target_genes',
#  'gp_source_genes','gp_target_genes','gp_source_genes_weights','gp_target_genes_weights',
#  'gp_source_genes_importances','gp_target_genes_importances']

In [ ]:
gp_summary_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 3.1 Cross-tab of niche vs cell type
ct = pd.crosstab(model.adata.obs[latent_cluster_key], model.adata.obs[cell_type_key], normalize="index")
plt.figure(figsize=(12, 6))
sns.heatmap(ct, cmap="hot_r", linewidths=0.0)
plt.title("Niche vs Cell Type (row-normalized)")
plt.xlabel("Cell type")
plt.ylabel("Niche")
plt.tight_layout()
plt.savefig(f"{figure_folder_path}/niche_vs_celltype_heatmap.pdf")
plt.show()


In [ ]:
loadings = pd. read_csv(f"{KB_DATA}/metadata/loadings_adata_LIANA_all_all_5.csv", index_col=0)

In [ ]:
loadings_factor2 = loadings[loadings['Factor'] == 'Factor2']

In [ ]:
loadings_factor2

In [ ]:
model.adata.obs_keys()

In [ ]:
import warnings

import scanpy as sc
import decoupler as dc

In [ ]:
hallmark = dc.op.hallmark(organism="human")
hallmark

In [ ]:
adata = dc.mt.ulm(data=model.adata, net=hallmark)

In [ ]:
adata

In [ ]:
score = dc.pp.get_obsm(adata=adata, key="score_ulm")
score

In [ ]:
df = dc.tl.rankby_group(adata=score, groupby=latent_cluster_key, reference="rest", method="t-test_overestim_var")
df = df[df["stat"] > 0]
df

In [ ]:
n_markers = 3
source_markers = (
    df.groupby("group")
    .head(n_markers)
    .drop_duplicates("name")
    .groupby("group")["name"]
    .apply(lambda x: list(x))
    .to_dict()
)
source_markers

In [ ]:
fig = sc.pl.matrixplot(
    adata=score,
    var_names=source_markers,
    groupby=latent_cluster_key,
    dendrogram=True,
    standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="RdBu_r",
    swap_axes=True,
    return_fig=True
)

fig.savefig(f"{KB_PROJECT}/niche_compass_results/artifacts/single_sample/02092025_162433/figures/hallmark_marker_pathways.pdf")

In [ ]:
list(dc.op.show_resources().name)

In [ ]:
markers = dc.op.resource(name="MSigDB")
markers

In [ ]:
markers.collection.unique()

In [ ]:
# Filter by canonical_marker and human
markers = markers[
    markers["collection"].astype(str) == "reactome_pathways"    
]

# Remove duplicated entries
markers = markers[~markers.duplicated(["geneset", "genesymbol"])]

markers = markers.rename(columns={"geneset": "source", "genesymbol": "target"})
markers = markers[["source", "target"]]
markers

In [ ]:
adata_molec = dc.mt.ulm(data=model.adata, net=markers)

In [ ]:
adata_molec

In [ ]:
score = dc.pp.get_obsm(adata=adata_molec, key="score_ulm")
score

In [ ]:
df = dc.tl.rankby_group(adata=score, groupby=latent_cluster_key, reference="rest", method="t-test_overestim_var")
df = df[df["stat"] > 0]
df

In [ ]:
n_markers = 3
source_markers = (
    df.groupby("group")
    .head(n_markers)
    .drop_duplicates("name")
    .groupby("group")["name"]
    .apply(lambda x: list(x))
    .to_dict()
)
source_markers

In [ ]:
fig = sc.pl.matrixplot(
    adata=score,
    var_names=source_markers,
    groupby=latent_cluster_key,
    dendrogram=True,
    standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="RdBu_r",
    swap_axes=True,
    return_fig=True
)

fig.savefig(f"{KB_PROJECT}/niche_compass_results/artifacts/single_sample/02092025_162433/figures/reactome_marker_pathways.pdf")

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import numpy as np


# Color map for niches
niche_colors = None
try:
    from nichecompass.utils import create_new_color_dict
    niche_colors = create_new_color_dict(adata=model.adata, cat_key=latent_cluster_key)
except Exception:
    pass

# Plot each TMA_ID in its own panel
tma_key = "TMA_ID"  # change if your column is named differently
tma_ids = model.adata.obs[tma_key].unique().tolist()

n_cols = 3
n_rows = int(np.ceil(len(tma_ids) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
axes = np.array(axes).reshape(n_rows, n_cols)

for i, tma in enumerate(tma_ids):
    r, c = divmod(i, n_cols)
    ax = axes[r, c]
    ad = model.adata[model.adata.obs[tma_key] == tma]
    sc.pl.spatial(
        adata=ad,
        color=[latent_cluster_key],
        palette=niche_colors,
        spot_size=spot_size,
        title=f"TMA_ID: {tma}",
        legend_loc=None,
        ax=ax,
        show=False
    )

# Remove any empty axes
for j in range(len(tma_ids), n_rows*n_cols):
    fig.delaxes(axes.flatten()[j])

plt.tight_layout()
plt.savefig(f"{figure_folder_path}/niches_per_TMA_ID.pdf", bbox_inches="tight")
plt.show()

In [ ]:
adata

In [ ]:
import numpy as np
import pandas as pd
import decoupler as dc
import seaborn as sns
import matplotlib.pyplot as plt

# ---- CONFIG ----
adata = model.adata

factors_to_score = ["Factor2"]
use_raw = False  # set True if you want to use adata.raw

cell_type_key = "predic"

# Your loadings must have columns: ["Factor","view","variable","Loading"]
# loadings = pd.read_csv("your_loadings.csv")

# Optional mapping to merge subtypes, e.g. {"IC-A":"IC","IC-B":"IC"}
unify_levels = None  # or a dict as above

adata.obs_names_make_unique()

def score_factor_for_view_dcmt(
    adata, loadings, celltype, factor,
    cell_type_key="view", unify_levels=None, use_raw=False
):
    # 1) Filter loadings for the factor
    loadings_factor = loadings[loadings["Factor"] == factor].copy()
    if loadings_factor.empty:
        return pd.DataFrame(index=pd.Index([], name="obs_names"))

    # 2) Subset AnnData for the desired celltype(s)
    if unify_levels and isinstance(celltype, str) and celltype in set(unify_levels.values()):
        # If passing unified label (e.g., "IC"), collect originals
        to_keep = [k for k, v in unify_levels.items() if v == celltype]
        sub = adata[adata.obs[cell_type_key].isin(to_keep)].copy()
        sub.obs[cell_type_key] = sub.obs[cell_type_key].replace(unify_levels)
        view_for_net = celltype
    elif isinstance(celltype, (list, tuple, set)):
        sub = adata[adata.obs[cell_type_key].isin(celltype)].copy()
        view_for_net = list(celltype)[0] if len(celltype) else None
    else:
        sub = adata[adata.obs[cell_type_key] == celltype].copy()
        view_for_net = celltype

    if sub.n_obs == 0:
        return pd.DataFrame(index=pd.Index([], name="obs_names"))

    # 3) Subset loadings to that view label
    net = loadings_factor[loadings_factor["view"] == view_for_net].copy()
    if net.empty:
        return pd.DataFrame(index=pd.Index([], name="obs_names"))

    # 4) Filter genes present
    net = net[net["variable"].isin(sub.var_names)].copy()
    if net.empty:
        return pd.DataFrame(index=pd.Index([], name="obs_names"))

    # 5) Drop empty (all-zero) cells
    X = sub.raw.X if (use_raw and sub.raw is not None) else sub.X
    if hasattr(X, "toarray"):
        nnz = np.array((X != 0).sum(axis=1)).ravel()
    else:
        nnz = (X != 0).sum(axis=1)
    non_empty = nnz > 0
    sub = sub[non_empty].copy()
    if sub.n_obs == 0:
        return pd.DataFrame(index=pd.Index([], name="obs_names"))

    # 6) Rename columns for decoupler's new API: source/target/weight
    net_std = net.rename(columns={"Factor": "source", "variable": "target", "Loading": "weight"})[
        ["source", "target", "weight"]
    ].drop_duplicates(subset=["source","target"])


    dc.mt.ulm(
        data=sub,
        net=net_std,
        tmin=1,
        raw=use_raw,
        verbose=False,
    )

    # decoupler writes scores to sub.obsm["ulm_estimate"] with columns = sources (i.e., Factor names)
    df = sub.obsm["score_ulm"]
    # Ensure we have the desired factor column
    col = factor if factor in df.columns else df.columns[0]
    out = df[[col]].copy()
    out.columns = [factor]
    out.index.name = "obs_names"
    return out

In [ ]:
views = adata.obs[cell_type_key].unique().tolist()

for factor in factors_to_score:
    per_view = []
    for view in views:
        view_to_use = view
        res_df = score_factor_for_view_dcmt(
            adata=adata,
            loadings=loadings,
            celltype=view_to_use,
            factor=factor,
            cell_type_key=cell_type_key,
            unify_levels=unify_levels,
            use_raw=use_raw
        )
        if res_df.shape[0] > 0:
            res_df["view"] = view_to_use
            per_view.append(res_df)

    if len(per_view) == 0:
        adata.obs[f"{factor}_ulm"] = np.nan
        continue

    results_allviews = pd.concat(per_view, axis=0)
    results_allviews = results_allviews[~results_allviews.index.duplicated(keep="first")]
    # Initialize column with NaN
    adata.obs[f"{factor}_ulm"] = np.nan
    common = adata.obs_names.intersection(results_allviews.index)
    adata.obs.loc[common, f"{factor}_ulm"] = results_allviews.loc[common, factor].astype(float).values

In [ ]:
factor = "Factor2_ulm"
if factor not in adata.obs.columns:
    raise KeyError(f"{factor} not found in adata.obs. Make sure the scoring above ran successfully.")

plot_df = adata.obs[[latent_cluster_key, cell_type_key, factor]].dropna().copy()

# Faceted boxes: niches on x, one facet per cell type
g = sns.catplot(
    data=plot_df,
    x=latent_cluster_key, y=factor,
    col=cell_type_key, col_wrap=4,  # tweak wrap
    kind="box",
    sharey=False, height=3.2, aspect=1.2,
    showfliers=False
)
g.set_titles("{col_name}")
g.set_xticklabels(rotation=45)
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle("Factor2 ULM score across niches (per cell type)")
plt.savefig(f"{figure_folder_path}/Factor2_ulm_box_by_niche_per_celltype.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact, binomtest
from statsmodels.stats.multitest import multipletests
import scanpy as sc

In [ ]:
# Set your keys:
condition_key = "condition"  # <-- change to your column name

# Quick checks
print("Conditions:", adata.obs[condition_key].unique().tolist())
print("Niches:", adata.obs[latent_cluster_key].unique().tolist())

In [ ]:
# Counts table
counts = pd.crosstab(adata.obs[latent_cluster_key], adata.obs[condition_key])
counts.index.name = "niche"
counts.columns.name = "condition"

# Proportions (row-normalized)
props = counts.div(counts.sum(axis=1), axis=0)

# Save
counts.to_csv(f"{figure_folder_path}/condition_counts_by_niche.csv")
props.to_csv(f"{figure_folder_path}/condition_proportions_by_niche.csv")

# Plot proportions heatmap
plt.figure(figsize=(max(8, 0.6*counts.shape[1]), max(4, 0.4*counts.shape[0])))
sns.heatmap(props, annot=True, fmt=".2f", cmap="mako", cbar_kws={"label":"Proportion"})
plt.title("Condition composition per niche (row-normalized)")
plt.xlabel("Condition")
plt.ylabel("Niche")
plt.tight_layout()
plt.savefig(f"{figure_folder_path}/condition_proportions_by_niche_heatmap.pdf", bbox_inches="tight")
plt.show()

# Stacked barplot
props_plot = props.reset_index().melt(id_vars="niche", var_name="condition", value_name="proportion")
plt.figure(figsize=(12, 6))
sns.barplot(data=props_plot, x="niche", y="proportion", hue="condition")
plt.title("Condition proportions per niche")
plt.ylabel("Proportion")
plt.xlabel("Niche")
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1,1))
plt.tight_layout()
plt.savefig(f"{figure_folder_path}/condition_proportions_by_niche_stacked.pdf", bbox_inches="tight")
plt.show()

In [ ]:
adata.obs.TMA_ID.unique()

In [ ]:
obs = adata.obs[[latent_cluster_key, condition_key]].copy()
all_counts = obs[condition_key].value_counts()
N_total = len(obs)

# Global baseline probability for each condition
p_global = (all_counts / N_total).to_dict()

records = []
for niche, sub in obs.groupby(latent_cluster_key):
    n = len(sub)
    cond_counts = sub[condition_key].value_counts()
    for cond, k in cond_counts.items():
        p0 = p_global[cond]
        # one-sided test: enrichment (probability of >= k successes)
        bt = binomtest(k=k, n=n, p=p0, alternative="greater")
        records.append({"niche": niche, "condition": cond, "n": n, "k": k, "p0": p0, "p_binom": bt.pvalue,
                        "prop": k/n if n>0 else np.nan,
                        "prop_enrichment": (k/n - p0) if n>0 else np.nan})

binom_df = pd.DataFrame(records)
# Multiple testing correction across all niche-condition pairs
binom_df["p_adj"] = multipletests(binom_df["p_binom"], method="fdr_bh")[1]
binom_df = binom_df.sort_values(["p_adj", "prop_enrichment"], ascending=[True, False])
binom_df.to_csv(f"{figure_folder_path}/condition_enrichment_binomial.csv", index=False)
binom_df.head(15)

In [ ]:
records = []
for niche, sub in obs.groupby(latent_cluster_key):
    n = len(sub)
    for cond in obs[condition_key].unique():
        k = (sub[condition_key] == cond).sum()
        # Build 2x2: in-niche vs out-of-niche, cond vs not-cond
        rest = obs[obs[latent_cluster_key] != niche]
        a = k
        b = n - k
        c = (rest[condition_key] == cond).sum()
        d = len(rest) - c
        # one-sided greater (enrichment in niche)
        oddsratio, p = fisher_exact([[a, b], [c, d]], alternative="greater")
        prop = k/n if n>0 else np.nan
        p0 = (c + a) / (len(rest) + n)
        records.append({"niche": niche, "condition": cond, "n": n, "k": k,
                        "oddsratio": oddsratio, "p_fisher": p,
                        "prop": prop, "p0": p0, "prop_enrichment": prop - p0})
fisher_df = pd.DataFrame(records)
fisher_df["p_adj"] = multipletests(fisher_df["p_fisher"], method="fdr_bh")[1]
fisher_df = fisher_df.sort_values(["p_adj", "oddsratio"], ascending=[True, False])
fisher_df.to_csv(f"{figure_folder_path}/condition_enrichment_fisher.csv", index=False)
fisher_df.head(15)

In [ ]:
# Use Fisher results for display (similar for binomial)
df = fisher_df.copy()
df["sign"] = np.sign(df["prop_enrichment"])
df["neglog10_fdr"] = -np.log10(df["p_adj"].clip(lower=1e-300))
df["signed_score"] = df["neglog10_fdr"] * df["sign"]

pivot = df.pivot(index="niche", columns="condition", values="signed_score").fillna(0)

plt.figure(figsize=(max(8, 0.6*pivot.shape[1]), max(4, 0.5*pivot.shape[0])))
sns.heatmap(pivot, cmap="coolwarm", center=0, cbar_kws={"label":"signed −log10(FDR)"})
plt.title("Condition enrichment per niche (Fisher one-sided, FDR)")
plt.xlabel("Condition")
plt.ylabel("Niche")
plt.tight_layout()
plt.savefig(f"{figure_folder_path}/condition_enrichment_signedlog10fdr_heatmap.pdf", bbox_inches="tight")
plt.show()

# List significant niche-condition pairs
sig = df[df["p_adj"] < 0.05].sort_values(["p_adj", "oddsratio"], ascending=[True, False])
sig.to_csv(f"{figure_folder_path}/condition_enriched_pairs_fdr05.csv", index=False)
sig.head(20)

In [ ]:
# Compute pivot as before
#df = fisher_df.copy()
df["sign"] = np.sign(df["prop_enrichment"])
df["neglog10_fdr"] = -np.log10(df["p_adj"].clip(lower=1e-300))
df["signed_score"] = df["neglog10_fdr"] * df["sign"]

pivot = df.pivot(index="niche", columns="condition", values="signed_score").fillna(0)

# Use clustermap instead of heatmap
g = sns.clustermap(
    pivot,
    cmap="coolwarm",
    center=0,
    row_cluster=True,    # set False to keep original row order
    col_cluster=True,    # set False to keep original column order
    figsize=(max(8, 0.6*pivot.shape[1]), max(6, 0.5*pivot.shape[0])),
    dendrogram_ratio=0.12,      # adjust dendrogram vs heatmap space
    cbar_pos=(0.02, 0.8, 0.03, 0.15),  # x, y, w, h (fraction of figure); tweak as needed
    cbar_kws={"label": "signed −log10(FDR)"},
    xticklabels=True,
    yticklabels=True,
    linewidths=0,  # optional
)

# Rotate x tick labels for readability
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha="right")
plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0)

# Title and axis labels (clustermap returns a Figure embedded in g.fig)
g.ax_heatmap.set_xlabel("Condition")
g.ax_heatmap.set_ylabel("Niche")
g.fig.suptitle("Condition enrichment per niche (Fisher one-sided, FDR)", y=1.02)

# Tight layout and save
g.fig.tight_layout()
g.fig.savefig(f"{figure_folder_path}/condition_enrichment_signedlog10fdr_clustermap.pdf",
              bbox_inches="tight")
plt.show()



In [ ]:
import re
import pandas as pd

# 1) Helper functions
def normalize_key(x: str) -> str | None:
    """Normalize keys to a canonical form for matching."""
    if x is None:
        return None
    x = str(x).strip()
    if x == "" or x.lower() == "na":
        return None
    # Remove spaces
    x = re.sub(r"\s+", "", x)
    # If pure K-number variants like K094 or k 94 → K94
    m = re.fullmatch(r"(?i)K0*(\d+)", x)
    if m:
        return f"K{int(m.group(1))}"
    # If Apollo variants: Apollot0_A01, apollo d0 a1, apollod0A1 → APOLLOT0_A1 / APOLLOD0_A1
    m = re.fullmatch(r"(?i)(APOLLO[TD]0)_?([A-H])0*(\d+)", x)
    if m:
        base = m.group(1).upper()        # APOLLOT0 or APOLLOD0
        well = f"{m.group(2).upper()}{int(m.group(3))}"  # A1, B6, etc.
        return f"{base}_{well}"
    # General cleanup and uppercase (keeps underscores)
    return x.upper()

def derive_join_key_from_tma(tma_id: str) -> str | None:
    """Derive a normalized join key from a TMA_ID string."""
    if not isinstance(tma_id, str):
        return None
    s = tma_id

    # 1) If there's a K-number anywhere, prefer that (e.g., Utrecht1 ... K26 TIN → K26)
    m = re.search(r"\bK\s*0*(\d+)\b", s, flags=re.IGNORECASE)
    if m:
        return normalize_key(f"K{int(m.group(1))}")

    # 2) Apollo t0/d0 + well code (letter+number), e.g., "Appollo t0 F6 CTRL" → Apollot0_F6
    stage = None
    if re.search(r"\bt\s*0\b", s, flags=re.IGNORECASE):
        stage = "APOLLOT0"
    elif re.search(r"\bd\s*0\b", s, flags=re.IGNORECASE):
        stage = "APOLLOD0"

    m = re.search(r"\b([A-H])\s*0*(\d+)\b", s, flags=re.IGNORECASE)
    if stage and m:
        well = f"{m.group(1).upper()}{int(m.group(2))}"
        return f"{stage}_{well}"

    return None

# 2) Read supplemental CSV (semicolon-separated, first row is a title)
supp_path = f"{KB_PROJECT}/niche_compass_results/Supplemental.csv"
meta = pd.read_csv(supp_path, sep=";", skiprows=1, dtype=str)

# Drop stray unnamed first column if present (looks like row numbers 1,2,... in your sample)
if meta.columns[0].startswith("Unnamed") or meta.columns[0] == "":
    meta = meta.drop(columns=meta.columns[0])

# Clean whitespace in column names and values
meta.columns = [c.strip() for c in meta.columns]
meta = meta.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# 3) Build an alias → row-index map from several possible columns
possible_alias_cols = [c for c in ["sample_ID", "ck_number_rna", "ck_number_atac", "patient_ID"] if c in meta.columns]
aliases = []
for ridx, row in meta.iterrows():
    for c in possible_alias_cols:
        alias = normalize_key(row[c])
        if alias is not None:
            aliases.append({"alias": alias, "meta_idx": ridx})

aliases_df = pd.DataFrame(aliases)
# Keep first occurrence if duplicates; warn if duplicates exist
dups = aliases_df["alias"].duplicated(keep=False)
if dups.any():
    dup_aliases = aliases_df.loc[dups, "alias"].unique().tolist()
    print(f"Warning: duplicate aliases in metadata for {len(dup_aliases)} keys. Using first occurrence:\n{dup_aliases[:20]}{' ...' if len(dup_aliases)>20 else ''}")

aliases_df = aliases_df.drop_duplicates(subset="alias", keep="first")
alias_to_metaidx = aliases_df.set_index("alias")["meta_idx"]

# 4) Derive join keys from adata.obs['TMA_ID'] and map to metadata rows
# Assumes you already have 'adata' loaded and adata.obs['TMA_ID'] present
def attach_supplemental_to_adata(adata, meta_df, alias_to_metaidx, prefix="supp_"):
    if "TMA_ID" not in adata.obs.columns:
        raise KeyError("adata.obs must contain a 'TMA_ID' column.")

    join_keys = adata.obs["TMA_ID"].apply(derive_join_key_from_tma)
    # Map to metadata row indices
    meta_idx_series = join_keys.map(alias_to_metaidx)

    # Build a DataFrame aligned to adata.obs.index with supplemental columns
    supp_cols = [c for c in meta_df.columns]  # keep all columns
    out = pd.DataFrame(index=adata.obs.index, columns=[prefix + c for c in supp_cols])

    mask = meta_idx_series.notna()
    if mask.any():
        # meta_df.loc[indices] will return rows in the order of indices
        matched_meta = meta_df.loc[meta_idx_series[mask].astype(int)].set_index(adata.obs.index[mask])
        out.loc[mask, :] = matched_meta.rename(columns=lambda c: prefix + c)

    # Attach to adata.obs
    adata.obs = pd.concat([adata.obs, out], axis=1)

    # Simple report
    n_matched = int(mask.sum())
    n_total = adata.n_obs
    print(f"Matched {n_matched}/{n_total} observations "
          f"({n_matched/n_total:.1%}). Unmatched: {n_total - n_matched}.")

    # Optionally, inspect a few unmatched TMA_IDs
    unmatched_examples = adata.obs.loc[~mask, "TMA_ID"].dropna().unique().tolist()[:20]
    if len(unmatched_examples) > 0:
        print("Examples of unmatched TMA_IDs (first 20):")
        for u in unmatched_examples:
            print(" -", u)

# 5) Run the attachment
# from anndata import AnnData  # if needed
# import scanpy as sc          # if using Scanpy
attach_supplemental_to_adata(adata, meta, alias_to_metaidx, prefix="supp_")

In [ ]:
adata

In [ ]:
adata.obs.supp_proteinuria_around_biopsy.unique()

In [ ]:
supp_egfr_at_time_of_biopsy

In [ ]:
obs = adata.obs[[latent_cluster_key, condition_key]].copy()
all_counts = obs[condition_key].value_counts()
N_total = len(obs)

# Global baseline probability for each condition
p_global = (all_counts / N_total).to_dict()

records = []
for niche, sub in obs.groupby(latent_cluster_key):
    n = len(sub)
    cond_counts = sub[condition_key].value_counts()
    for cond, k in cond_counts.items():
        p0 = p_global[cond]
        # one-sided test: enrichment (probability of >= k successes)
        bt = binomtest(k=k, n=n, p=p0, alternative="greater")
        records.append({"niche": niche, "condition": cond, "n": n, "k": k, "p0": p0, "p_binom": bt.pvalue,
                        "prop": k/n if n>0 else np.nan,
                        "prop_enrichment": (k/n - p0) if n>0 else np.nan})

binom_df = pd.DataFrame(records)
# Multiple testing correction across all niche-condition pairs
binom_df["p_adj"] = multipletests(binom_df["p_binom"], method="fdr_bh")[1]
binom_df = binom_df.sort_values(["p_adj", "prop_enrichment"], ascending=[True, False])
binom_df.to_csv(f"{figure_folder_path}/condition_enrichment_binomial.csv", index=False)
binom_df.head(15)

records = []
for niche, sub in obs.groupby(latent_cluster_key):
    n = len(sub)
    for cond in obs[condition_key].unique():
        k = (sub[condition_key] == cond).sum()
        # Build 2x2: in-niche vs out-of-niche, cond vs not-cond
        rest = obs[obs[latent_cluster_key] != niche]
        a = k
        b = n - k
        c = (rest[condition_key] == cond).sum()
        d = len(rest) - c
        # one-sided greater (enrichment in niche)
        oddsratio, p = fisher_exact([[a, b], [c, d]], alternative="greater")
        prop = k/n if n>0 else np.nan
        p0 = (c + a) / (len(rest) + n)
        records.append({"niche": niche, "condition": cond, "n": n, "k": k,
                        "oddsratio": oddsratio, "p_fisher": p,
                        "prop": prop, "p0": p0, "prop_enrichment": prop - p0})
fisher_df = pd.DataFrame(records)
fisher_df["p_adj"] = multipletests(fisher_df["p_fisher"], method="fdr_bh")[1]
fisher_df = fisher_df.sort_values(["p_adj", "oddsratio"], ascending=[True, False])
fisher_df.to_csv(f"{figure_folder_path}/condition_enrichment_fisher.csv", index=False)
fisher_df.head(15)

# Use Fisher results for display (similar for binomial)
df = fisher_df.copy()
df["sign"] = np.sign(df["prop_enrichment"])
df["neglog10_fdr"] = -np.log10(df["p_adj"].clip(lower=1e-300))
df["signed_score"] = df["neglog10_fdr"] * df["sign"]

pivot = df.pivot(index="niche", columns="condition", values="signed_score").fillna(0)

plt.figure(figsize=(max(8, 0.6*pivot.shape[1]), max(4, 0.5*pivot.shape[0])))
sns.heatmap(pivot, cmap="coolwarm", center=0, cbar_kws={"label":"signed −log10(FDR)"})
plt.title("Condition enrichment per niche (Fisher one-sided, FDR)")
plt.xlabel("Condition")
plt.ylabel("Niche")
plt.tight_layout()
plt.savefig(f"{figure_folder_path}/condition_enrichment_signedlog10fdr_heatmap.pdf", bbox_inches="tight")
plt.show()

# List significant niche-condition pairs
sig = df[df["p_adj"] < 0.05].sort_values(["p_adj", "oddsratio"], ascending=[True, False])
sig.to_csv(f"{figure_folder_path}/condition_enriched_pairs_fdr05.csv", index=False)
sig.head(20)

# Compute pivot as before
#df = fisher_df.copy()
df["sign"] = np.sign(df["prop_enrichment"])
df["neglog10_fdr"] = -np.log10(df["p_adj"].clip(lower=1e-300))
df["signed_score"] = df["neglog10_fdr"] * df["sign"]

pivot = df.pivot(index="niche", columns="condition", values="signed_score").fillna(0)

# Use clustermap instead of heatmap
g = sns.clustermap(
    pivot,
    cmap="coolwarm",
    center=0,
    row_cluster=True,    # set False to keep original row order
    col_cluster=True,    # set False to keep original column order
    figsize=(max(8, 0.6*pivot.shape[1]), max(6, 0.5*pivot.shape[0])),
    dendrogram_ratio=0.12,      # adjust dendrogram vs heatmap space
    cbar_pos=(0.02, 0.8, 0.03, 0.15),  # x, y, w, h (fraction of figure); tweak as needed
    cbar_kws={"label": "signed −log10(FDR)"},
    xticklabels=True,
    yticklabels=True,
    linewidths=0,  # optional
)

# Rotate x tick labels for readability
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha="right")
plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0)

# Title and axis labels (clustermap returns a Figure embedded in g.fig)
g.ax_heatmap.set_xlabel("Condition")
g.ax_heatmap.set_ylabel("Niche")
g.fig.suptitle("Condition enrichment per niche (Fisher one-sided, FDR)", y=1.02)

# Tight layout and save
g.fig.tight_layout()
g.fig.savefig(f"{figure_folder_path}/condition_enrichment_signedlog10fdr_clustermap.pdf",
              bbox_inches="tight")
plt.show()



In [ ]:
# Imports
import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import binomtest, fisher_exact
from statsmodels.stats.multitest import multipletests

# -----------------------------
# 1) Clean supplemental IFTA and eGFR into numeric columns
# -----------------------------
def parse_ifta_to_pct(x, treat_one_as_percent=False):
    """
    Convert IFTA to percent:
      - '<5' or '≤5' → slightly below 5 (4.999) so it bins into <5% with right=False.
      - numeric > 1     → already percent (e.g., 10 → 10%).
      - numeric < 1     → fraction of 1 (e.g., 0.45 → 45%).
      - numeric == 1    → 1% if treat_one_as_percent=True else 100%? We default to 1% (more plausible).
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    s = str(x).strip()
    if s == "" or s.lower() == "na":
        return np.nan
    s = s.replace(",", ".").replace("%", "")

    # Handles "<5", "≤5", "< 5.0"
    m = re.match(r"^\s*[<≤]\s*(\d+(\.\d+)?)\s*$", s)
    if m:
        val = float(m.group(1))
        return max(val - 1e-3, 0.0)  # slightly less than the bound

    # Plain number
    try:
        val = float(s)
    except Exception:
        return np.nan

    # Apply your rule
    if val > 1.0:
        pct = val
    elif val < 1.0:
        pct = val * 100.0
    else:  # val == 1.0
        pct = 1.0 if treat_one_as_percent else 1.0  # default 1% (change to 100.0 if you disagree)

    # Sanity: clip to [0, 100] if needed (or set to NaN if you prefer strict)
    if not np.isfinite(pct) or pct < 0 or pct > 100:
        return np.nan
    return pct

def parse_numeric(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    s = str(x).strip()
    if s == "" or s.lower() == "na":
        return np.nan
    s = s.replace(",", ".")
    try:
        return float(s)
    except Exception:
        return np.nan

# Expect these columns from your previous merge step
if "supp_IFTA" in adata.obs.columns:
    adata.obs["IFTA_pct"] = adata.obs["supp_IFTA"].apply(parse_ifta_to_pct)
else:
    # Fallback if you named it differently
    adata.obs["IFTA_pct"] = np.nan

if "supp_egfr_at_time_of_biopsy" in adata.obs.columns:
    adata.obs["EGFR"] = adata.obs["supp_egfr_at_time_of_biopsy"].apply(parse_numeric)
else:
    adata.obs["EGFR"] = np.nan

# -----------------------------
# 2) Binning
# -----------------------------
ifta_bins = [-np.inf, 5, 25, 50, np.inf]
ifta_labels = ['<5%', '5%-25%', '25%-50%', '>=50%']
adata.obs['ifta_group'] = pd.cut(
    adata.obs['IFTA_pct'],
    bins=ifta_bins,
    labels=ifta_labels,
    right=False,
    include_lowest=True
)

egfr_bins = [-np.inf, 30, 60, 90, np.inf]
egfr_labels = ['<30', '30-60', '60-90', '>90']
adata.obs['egfr_group'] = pd.cut(
    adata.obs['EGFR'],
    bins=egfr_bins,
    labels=egfr_labels,
    right=False,
    include_lowest=True
)

# Ensure categorical dtype (helps with plotting and consistent ordering)
adata.obs['ifta_group'] = adata.obs['ifta_group'].cat.set_categories(ifta_labels, ordered=True)
adata.obs['egfr_group'] = adata.obs['egfr_group'].cat.set_categories(egfr_labels, ordered=True)

# -----------------------------
# 3) Enrichment tests (cell-level, as in your code)
# -----------------------------
def run_enrichment_cell_level(adata, latent_cluster_key, condition_key, figure_folder_path, prefix):
    obs = adata.obs[[latent_cluster_key, condition_key]].dropna().copy()
    # Drop NAs in condition_key; ensure strings
    obs[condition_key] = obs[condition_key].astype(str)

    # Global baseline
    all_counts = obs[condition_key].value_counts()
    N_total = len(obs)
    p_global = (all_counts / N_total).to_dict()

    # Binomial
    records = []
    for niche, sub in obs.groupby(latent_cluster_key):
        n = len(sub)
        cond_counts = sub[condition_key].value_counts()
        for cond, k in cond_counts.items():
            p0 = p_global.get(cond, 0.0)
            bt = binomtest(k=k, n=n, p=p0, alternative="greater")
            records.append({
                "niche": niche, "condition": cond, "n": n, "k": k, "p0": p0,
                "p_binom": bt.pvalue,
                "prop": k/n if n>0 else np.nan,
                "prop_enrichment": (k/n - p0) if n>0 else np.nan
            })
    binom_df = pd.DataFrame(records)
    if not binom_df.empty:
        binom_df["p_adj"] = multipletests(binom_df["p_binom"], method="fdr_bh")[1]
        binom_df = binom_df.sort_values(["p_adj", "prop_enrichment"], ascending=[True, False])

    os.makedirs(figure_folder_path, exist_ok=True)
    binom_path = os.path.join(figure_folder_path, f"{prefix}_enrichment_binomial.csv")
    binom_df.to_csv(binom_path, index=False)

    # Fisher
    records = []
    for niche, sub in obs.groupby(latent_cluster_key):
        n = len(sub)
        rest = obs[obs[latent_cluster_key] != niche]
        for cond in obs[condition_key].unique():
            k = (sub[condition_key] == cond).sum()
            a = k
            b = n - k
            c = (rest[condition_key] == cond).sum()
            d = len(rest) - c
            oddsratio, p = fisher_exact([[a, b], [c, d]], alternative="greater")
            prop = k/n if n>0 else np.nan
            p0 = (c + a) / (len(rest) + n) if (len(rest)+n) > 0 else np.nan
            records.append({
                "niche": niche, "condition": cond, "n": n, "k": k,
                "oddsratio": oddsratio, "p_fisher": p,
                "prop": prop, "p0": p0, "prop_enrichment": prop - p0
            })
    fisher_df = pd.DataFrame(records)
    if not fisher_df.empty:
        fisher_df["p_adj"] = multipletests(fisher_df["p_fisher"], method="fdr_bh")[1]
        fisher_df = fisher_df.sort_values(["p_adj", "oddsratio"], ascending=[True, False])

    fisher_path = os.path.join(figure_folder_path, f"{prefix}_enrichment_fisher.csv")
    fisher_df.to_csv(fisher_path, index=False)

    # Heatmap/clustermap
    df = fisher_df.copy()
    if not df.empty:
        df["sign"] = np.sign(df["prop_enrichment"])
        df["neglog10_fdr"] = -np.log10(df["p_adj"].clip(lower=1e-300))
        df["signed_score"] = df["neglog10_fdr"] * df["sign"]
        pivot = df.pivot(index="niche", columns="condition", values="signed_score").fillna(0)

        plt.figure(figsize=(max(8, 0.6*pivot.shape[1]), max(4, 0.5*pivot.shape[0])))
        sns.heatmap(pivot, cmap="coolwarm", center=0, cbar_kws={"label":"signed −log10(FDR)"})
        plt.title(f"{prefix}: condition enrichment per niche (Fisher one-sided, FDR)")
        plt.xlabel("Condition")
        plt.ylabel("Niche")
        plt.tight_layout()
        plt.savefig(os.path.join(figure_folder_path, f"{prefix}_signedlog10fdr_heatmap.pdf"), bbox_inches="tight")
        plt.show()

        g = sns.clustermap(
            pivot,
            cmap="coolwarm",
            center=0,
            row_cluster=True,
            col_cluster=True,
            figsize=(max(8, 0.6*pivot.shape[1]), max(6, 0.5*pivot.shape[0])),
            dendrogram_ratio=0.12,
            cbar_pos=(0.02, 0.8, 0.03, 0.15),
            cbar_kws={"label": "signed −log10(FDR)"},
            xticklabels=True,
            yticklabels=True,
            linewidths=0,
        )
        plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha="right")
        plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0)
        g.ax_heatmap.set_xlabel("Condition")
        g.ax_heatmap.set_ylabel("Niche")
        g.fig.suptitle(f"{prefix}: condition enrichment per niche (Fisher one-sided, FDR)", y=1.02)
        g.fig.tight_layout()
        g.fig.savefig(os.path.join(figure_folder_path, f"{prefix}_signedlog10fdr_clustermap.pdf"), bbox_inches="tight")
        plt.show()

    # Significant pairs
    sig = df[df["p_adj"] < 0.05].sort_values(["p_adj", "oddsratio"], ascending=[True, False]) if not df.empty else df
    sig.to_csv(os.path.join(figure_folder_path, f"{prefix}_enriched_pairs_fdr05.csv"), index=False)
    return binom_df, fisher_df, sig

# -----------------------------
# 4) OPTIONAL: Sample-level enrichment (avoid cell-level pseudoreplication)
# -----------------------------
def run_enrichment_sample_presence(adata, latent_cluster_key, condition_key, tma_key, figure_folder_path, prefix):
    """
    For each niche, consider the set of TMA_IDs that contribute at least 1 cell (presence/absence),
    and test if the fraction of samples from each condition is enriched vs the global sample fraction.
    """
    obs = adata.obs[[latent_cluster_key, condition_key, tma_key]].dropna().copy()
    obs[condition_key] = obs[condition_key].astype(str)

    # One row per (TMA_ID, niche) with presence=1
    presence = (obs.groupby([tma_key, latent_cluster_key]).size() > 0).reset_index(name="present").query("present").drop(columns="present")

    # Condition per TMA_ID (assumes one condition per TMA_ID)
    cond_per_sample = obs[[tma_key, condition_key]].drop_duplicates(subset=tma_key)
    pres = presence.merge(cond_per_sample, on=tma_key, how="left")

    # Global sample baseline
    sample_counts = cond_per_sample[condition_key].value_counts()
    N_samples = len(cond_per_sample)
    p_global = (sample_counts / N_samples).to_dict()

    # Fisher per niche using counts of samples
    records = []
    for niche, sub in pres.groupby(latent_cluster_key):
        # Samples that have this niche
        in_samples = sub[tma_key].unique()
        out_samples = np.setdiff1d(cond_per_sample[tma_key].values, in_samples)

        in_cond = cond_per_sample[cond_per_sample[tma_key].isin(in_samples)]
        out_cond = cond_per_sample[cond_per_sample[tma_key].isin(out_samples)]

        for cond in cond_per_sample[condition_key].unique():
            a = (in_cond[condition_key] == cond).sum()
            b = len(in_cond) - a
            c = (out_cond[condition_key] == cond).sum()
            d = len(out_cond) - c
            oddsratio, p = fisher_exact([[a, b], [c, d]], alternative="greater")
            prop = a / len(in_cond) if len(in_cond) > 0 else np.nan
            p0 = p_global.get(cond, np.nan)
            records.append({
                "niche": niche, "condition": cond,
                "a_in_cond": a, "b_in_other": b, "c_out_cond": c, "d_out_other": d,
                "n_in": len(in_cond), "n_out": len(out_cond),
                "oddsratio": oddsratio, "p_fisher": p,
                "prop": prop, "p0": p0, "prop_enrichment": prop - p0 if pd.notna(prop) and pd.notna(p0) else np.nan
            })
    fisher_df = pd.DataFrame(records)
    if not fisher_df.empty:
        fisher_df["p_adj"] = multipletests(fisher_df["p_fisher"], method="fdr_bh")[1]
        fisher_df = fisher_df.sort_values(["p_adj", "oddsratio"], ascending=[True, False])

    os.makedirs(figure_folder_path, exist_ok=True)
    fisher_df.to_csv(os.path.join(figure_folder_path, f"{prefix}_sample_presence_fisher.csv"), index=False)
    return fisher_df

# -----------------------------
# 5) Per-niche dot/box/violin plots with one point per TMA_ID, colored by condition
# -----------------------------
def niche_sample_long_df(adata, niche_col, tma_col, value_cols, color_by, min_cells_per_niche=1):
    """
    Build a long DF with one row per (niche, TMA_ID) that has >= min_cells_per_niche cells in the niche.
    Adds unique sample-level metadata columns (value_cols, color_by).
    """
    # Count cells per (TMA_ID, niche)
    counts = adata.obs.groupby([tma_col, niche_col]).size().reset_index(name="cell_count")
    counts = counts[counts["cell_count"] >= min_cells_per_niche]

    # Unique sample-level info: assume value_cols and color_by do not vary within TMA_ID
    cols = [c for c in [tma_col, color_by] + value_cols if c in adata.obs.columns]
    sample_meta = adata.obs[cols].drop_duplicates(subset=tma_col)

    df = counts.merge(sample_meta, on=tma_col, how="left")

    # Long format for multiple value columns
    df_long = df.melt(
        id_vars=[niche_col, tma_col, "cell_count", color_by],
        value_vars=[c for c in value_cols if c in df.columns],
        var_name="measure", value_name="value"
    )
    # Drop NaNs in value
    df_long = df_long[pd.notna(df_long["value"])]
    return df_long

def plot_niche_sample_scatter(df_long, niche_col, color_by, figure_folder_path, prefix, kind="violin+swarm"):
    """
    kind options:
      - 'strip'
      - 'swarm'
      - 'box+strip'
      - 'violin+swarm' (default)
    """
    os.makedirs(figure_folder_path, exist_ok=True)

    for measure, sub in df_long.groupby("measure"):
        n_niches = sub[niche_col].nunique()
        plt.figure(figsize=(max(8, 0.6*n_niches), 5))

        if kind == "strip":
            sns.stripplot(data=sub, x=niche_col, y="value", hue=color_by, dodge=True, jitter=0.25)
        elif kind == "swarm":
            sns.swarmplot(data=sub, x=niche_col, y="value", hue=color_by, dodge=True)
        elif kind == "box+strip":
            sns.boxplot(data=sub, x=niche_col, y="value", showcaps=True, boxprops=dict(alpha=0.5))
            sns.stripplot(data=sub, x=niche_col, y="value", hue=color_by, dodge=True, jitter=0.2)
        else:  # violin+swarm
            sns.violinplot(data=sub, x=niche_col, y="value", inner=None, color="lightgray")
            sns.swarmplot(data=sub, x=niche_col, y="value", hue=color_by, dodge=True)

        plt.title(f"{measure} per niche (one dot per TMA_ID)")
        plt.xlabel("Niche")
        plt.ylabel(measure)
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        out = os.path.join(figure_folder_path, f"{prefix}_{measure}_per_niche_{kind}.pdf")
        plt.savefig(out, bbox_inches="tight")
        plt.show()

# -----------------------------
# 6) Run everything
# -----------------------------
# Keys (edit to your column names)

tma_key = "TMA_ID"                              # your TMA/sample column
figure_folder_path = f"{KB_PROJECT}/niche_compass_results/artifacts/single_sample/02092025_162433/figures/figures_enrichment"     # change as needed

# Choose which condition to color points by in plots (e.g., 'supp_disease' or 'supp_classification')
color_by = "supp_disease" if "supp_disease" in adata.obs.columns else "ifta_group"

# Enrichment on bins (cell-level, as in your code)
_ = run_enrichment_cell_level(adata, latent_cluster_key, "ifta_group", figure_folder_path, prefix="IFTA_group")
_ = run_enrichment_cell_level(adata, latent_cluster_key, "egfr_group", figure_folder_path, prefix="EGFR_group")

# OPTIONAL: sample-level enrichment (recommended to report alongside)
_ = run_enrichment_sample_presence(adata, latent_cluster_key, "ifta_group", tma_key, figure_folder_path, prefix="IFTA_group")
_ = run_enrichment_sample_presence(adata, latent_cluster_key, "egfr_group", tma_key, figure_folder_path, prefix="EGFR_group")

# Per-niche dot/violin for continuous values (one dot per TMA_ID, colored by condition)
df_long = niche_sample_long_df(
    adata,
    niche_col=latent_cluster_key,
    tma_col=tma_key,
    value_cols=["EGFR", "IFTA_pct"],
    color_by=color_by,
    min_cells_per_niche=1  # increase (e.g., 20) to require stronger presence per niche
)
plot_niche_sample_scatter(
    df_long,
    niche_col=latent_cluster_key,
    color_by=color_by,
    figure_folder_path=figure_folder_path,
    prefix="continuous",
    kind="violin+swarm"  # try 'box+strip' if you prefer boxplots
)

# -----------------------------
# 7) Quick sanity checks
# -----------------------------
print("IFTA raw uniques:", pd.Series(adata.obs.get("supp_IFTA")).unique()[:20])
print("IFTA_pct summary:", adata.obs["IFTA_pct"].describe())
print("EGFR summary:", adata.obs["EGFR"].describe())
print("ifta_group counts:\n", adata.obs["ifta_group"].value_counts(dropna=False))
print("egfr_group counts:\n", adata.obs["egfr_group"].value_counts(dropna=False))

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# -----------------------------
# Utilities to load fisher_df saved earlier
# -----------------------------
def load_fisher_df(figure_folder_path, prefix):
    """
    Tries to load the Fisher results you saved, e.g.:
      figure_folder_path/{prefix}_enrichment_fisher.csv
    Returns a DataFrame with columns including:
      niche, condition, n, k, oddsratio, p_fisher, p_adj, prop, p0, prop_enrichment
    """
    path = os.path.join(figure_folder_path, f"{prefix}_enrichment_fisher.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Cannot find {path}. Please provide the correct figure_folder_path/prefix.")
    df = pd.read_csv(path)
    # Basic hygiene
    for col in ["prop_enrichment", "prop", "oddsratio", "p_adj", "k", "n"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

# -----------------------------
# 1) Bubble map: effect + significance + counts
# -----------------------------
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import get_cmap

def plot_bubble_enrichment(fisher_df, out_path, title=None, min_n=10):
    """
    One point per (niche, condition):
      - color: signed −log10(FDR)
      - size: k (count of cells in niche with that condition)
      - transparency: greyed out if niche n < min_n (via overlay)
    """
    df = fisher_df.copy()
    if df.empty:
        print("fisher_df is empty. Skipping bubble plot.")
        return

    # Scores
    df["neglog10_fdr"] = -np.log10(df["p_adj"].astype(float).clip(lower=1e-300))
    df["sign"] = np.sign(df["prop_enrichment"].astype(float))
    df["score"] = df["neglog10_fdr"] * df["sign"]
    df["k"] = pd.to_numeric(df["k"], errors="coerce")
    df["n"] = pd.to_numeric(df["n"], errors="coerce")

    # Order axes by average signed score for readability
    row_order = df.groupby("niche")["score"].mean().sort_values(ascending=False).index.tolist()
    col_order = df.groupby("condition")["score"].mean().sort_values(ascending=False).index.tolist()
    df["niche"] = pd.Categorical(df["niche"], categories=row_order, ordered=True)
    df["condition"] = pd.Categorical(df["condition"], categories=col_order, ordered=True)

    # Color normalization centered at 0
    vmax = max(1.0, np.nanquantile(df["neglog10_fdr"], 0.98))
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
    cmap = get_cmap("coolwarm")

    # Split by n for alpha control
    df_large = df[df["n"] >= min_n]
    df_small = df[df["n"] < min_n]

    plt.figure(figsize=(max(8, 0.55*len(col_order)), max(4, 0.45*len(row_order))))
    ax = plt.gca()

    # Small-n (faint) plotted first
    if not df_small.empty:
        sns.scatterplot(
            data=df_small, x="condition", y="niche",
            hue="score", palette="coolwarm", hue_norm=norm,
            size="k", sizes=(10, 500),
            linewidth=0.5, edgecolor="k", alpha=0.3,
            legend=False, ax=ax
        )

    # Large-n (solid) on top
    sc_ax = sns.scatterplot(
        data=df_large, x="condition", y="niche",
        hue="score", palette="coolwarm", hue_norm=norm,
        size="k", sizes=(10, 500),
        linewidth=0.5, edgecolor="k", alpha=1.0,
        legend="brief", ax=ax
    )

    # Turn off hue legend (we’ll use a colorbar instead); keep size legend
    handles, labels = sc_ax.get_legend_handles_labels()
    # Seaborn legend packs both 'score' and 'k' levels; keep size items only
    keep = []
    if "k" in labels:
        # Find where size legend starts
        start = labels.index("k")
        keep = list(range(start, len(labels)))
    else:
        # If seaborn version differs, just drop legend and we’ll skip size legend
        keep = []

    if keep:
        new_handles = [handles[i] for i in keep]
        new_labels = [labels[i] for i in keep]
        ax.legend(new_handles, new_labels, title="k (count)", bbox_to_anchor=(1.02, 1), loc="upper left")
    else:
        ax.legend_.remove() if ax.get_legend() is not None else None

    # Add a proper colorbar and attach it to this axes
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("signed −log10(FDR)")

    # Optional: significance stars for FDR < 0.05
    sig = df[df["p_adj"] < 0.05]
    for _, r in sig.iterrows():
        ax.text(r["condition"], r["niche"], "*", ha="center", va="center", fontsize=10, color="black")

    ax.set_title(title or "Enrichment bubble map (color: signed −log10 FDR, size: k)")
    ax.set_xlabel("Condition")
    ax.set_ylabel("Niche")
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.show()

# -----------------------------
# 2) Stacked proportion bars per niche
# -----------------------------
def plot_stacked_proportions(adata, latent_cluster_key, condition_key, out_path, order_conditions=None, min_total=20):
    """
    Stacked bars of within-niche proportions for each condition.
    Marks niches with very low total counts (n < min_total) in the title.
    """
    obs = adata.obs[[latent_cluster_key, condition_key]].dropna().copy()
    obs[condition_key] = obs[condition_key].astype(str)

    counts = obs.groupby([latent_cluster_key, condition_key]).size().reset_index(name="count")
    totals = counts.groupby(latent_cluster_key)["count"].sum().reset_index(name="total")
    props = counts.merge(totals, on=latent_cluster_key)
    props["prop"] = props["count"] / props["total"]

    # Order
    niche_order = totals.sort_values("total", ascending=False)[latent_cluster_key].tolist()
    if order_conditions is None:
        # Order conditions by global frequency
        order_conditions = obs[condition_key].value_counts().index.tolist()

    props[latent_cluster_key] = pd.Categorical(props[latent_cluster_key], categories=niche_order, ordered=True)
    props[condition_key] = pd.Categorical(props[condition_key], categories=order_conditions, ordered=True)

    plt.figure(figsize=(max(8, 0.45*len(niche_order)), 5))
    bottom = np.zeros(len(niche_order))
    for cond in order_conditions:
        sub = props[props[condition_key] == cond].set_index(latent_cluster_key).reindex(niche_order)
        vals = sub["prop"].values
        plt.bar(niche_order, vals, bottom=bottom, label=str(cond))
        bottom += np.nan_to_num(vals)

    # Mark small-n niches
    small = totals[totals["total"] < min_total][latent_cluster_key].tolist()
    if small:
        plt.title(f"Within-niche composition of {condition_key} (warning: low n in {len(small)} niches)")

    plt.ylabel("Proportion")
    plt.xlabel("Niche")
    plt.xticks(rotation=45, ha="right")
    plt.legend(title=condition_key, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.show()

# -----------------------------
# 3) Effect-size heatmap (prop − global baseline)
# -----------------------------
def plot_effectsize_heatmap(adata, latent_cluster_key, condition_key, out_path):
    """
    Heatmap of (within-niche proportion − global proportion) for each condition.
    Positive = enriched, negative = depleted. No p-values, focuses on effect size.
    """
    obs = adata.obs[[latent_cluster_key, condition_key]].dropna().copy()
    obs[condition_key] = obs[condition_key].astype(str)

    # Global baseline
    global_counts = obs[condition_key].value_counts()
    p0 = (global_counts / len(obs)).to_dict()

    # Within-niche proportions
    counts = obs.groupby([latent_cluster_key, condition_key]).size().unstack(fill_value=0)
    props = counts.div(counts.sum(axis=1), axis=0)

    # Align to global keys
    props = props.reindex(columns=global_counts.index, fill_value=0)
    effect = props - pd.Series(p0)

    plt.figure(figsize=(max(8, 0.6*effect.shape[1]), max(4, 0.5*effect.shape[0])))
    sns.heatmap(effect, cmap="coolwarm", center=0, cbar_kws={"label": "prop − global"})
    plt.title(f"Effect size heatmap for {condition_key}")
    plt.xlabel("Condition")
    plt.ylabel("Niche")
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.show()

# -----------------------------
# 4) ECDFs for continuous values (cell-level or sample-level)
# -----------------------------
def plot_ecdf_by_niche(adata, latent_cluster_key, value_key, out_path, sample_level=False, tma_key="TMA_ID"):
    """
    ECDF per niche for a continuous measure (EGFR, IFTA_pct).
    - sample_level=True: one value per TMA (median), to avoid cell-level pseudoreplication.
    """
    df = adata.obs[[latent_cluster_key, value_key]].dropna().copy()
    if sample_level and tma_key in adata.obs.columns:
        tmp = adata.obs[[latent_cluster_key, tma_key, value_key]].dropna().copy()
        df = tmp.groupby([latent_cluster_key, tma_key])[value_key].median().reset_index()
        df = df[[latent_cluster_key, value_key]]

    n_niches = df[latent_cluster_key].nunique()
    plt.figure(figsize=(8, max(4, 0.35*n_niches)))
    sns.ecdfplot(data=df, x=value_key, hue=latent_cluster_key, linewidth=1.5)
    plt.title(f"ECDF of {value_key} by niche" + (" (sample-level)" if sample_level else " (cell-level)"))
    plt.xlabel(value_key)
    plt.ylabel("ECDF")
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.show()

# -----------------------------
# 5) Ridgeline (joy) plot for continuous values across niches (sample-level)
# -----------------------------
def plot_ridgeline_by_niche(adata, latent_cluster_key, value_key, out_path, tma_key="TMA_ID", sort_by="median"):
    """
    Ridgeline plot with one KDE per niche, using sample-level medians (one per TMA_ID per niche).
    """
    # Build sample-level values
    tmp = adata.obs[[latent_cluster_key, tma_key, value_key]].dropna().copy()
    df = tmp.groupby([latent_cluster_key, tma_key])[value_key].median().reset_index()

    # Order niches
    if sort_by == "median":
        order = df.groupby(latent_cluster_key)[value_key].median().sort_values().index.tolist()
    else:
        order = sorted(df[latent_cluster_key].unique())

    plt.figure(figsize=(8, max(4, 0.5*len(order))))
    y_offsets = {n: i for i, n in enumerate(order)}
    x = df[value_key].values

    # Plot per niche
    for niche in order:
        sub = df[df[latent_cluster_key] == niche][value_key].dropna().values
        if len(sub) < 2:
            # fall back to rug only
            y0 = y_offsets[niche]
            plt.plot(sub, np.full_like(sub, y0), "|", color="k", markersize=10)
            continue
        sns.kdeplot(sub, fill=True, alpha=0.6, linewidth=1.0, bw_method="scott", clip=(np.nanmin(sub), np.nanmax(sub)), label=None)
        # Move the last drawn collection to y-offset
        coll = plt.gca().collections[-1]
        # Transform vertices upward: not trivial; simplest: draw with transform.
        # Instead, re-draw using line/area with offset:
        # We'll approximate by overlaying a transparent fill using normalized y-scale
        # Simpler approach: plot horizontally stacked violins per niche
    plt.clf()

    # Simpler and robust: horizontal violins (sample-level)
    plt.figure(figsize=(8, max(4, 0.5*len(order))))
    sns.violinplot(
        data=df,
        y=pd.Categorical(df[latent_cluster_key], categories=order, ordered=True),
        x=value_key,
        inner="quartile", cut=0, scale="width", linewidth=0.8
    )
    sns.stripplot(
        data=df,
        y=pd.Categorical(df[latent_cluster_key], categories=order, ordered=True),
        x=value_key,
        color="k", size=2, alpha=0.6
    )
    plt.xlabel(value_key)
    plt.ylabel("Niche")
    plt.title(f"{value_key} per niche (sample-level violins)")
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.show()

# -----------------------------
# 6) Lollipop (Cleveland) plot: deviation from global baseline for each condition within a niche
# -----------------------------
def plot_lollipop_diffs(adata, latent_cluster_key, condition_key, niche_of_interest, out_path, top_n=20):
    """
    For a chosen niche, plot (within-niche proportion − global proportion) across conditions.
    """
    obs = adata.obs[[latent_cluster_key, condition_key]].dropna().copy()
    obs[condition_key] = obs[condition_key].astype(str)

    # Global baseline
    p0 = (obs[condition_key].value_counts() / len(obs))

    # In-niche
    sub = obs[obs[latent_cluster_key] == niche_of_interest]
    p_in = (sub[condition_key].value_counts() / len(sub))

    # Align
    df = pd.DataFrame({"p_in": p_in, "p0": p0}).fillna(0.0)
    df["diff"] = df["p_in"] - df["p0"]
    df = df.reindex(df["diff"].abs().sort_values(ascending=False).index)
    if top_n is not None:
        df = df.head(top_n)

    plt.figure(figsize=(8, max(3.5, 0.35*len(df))))
    y = np.arange(len(df))[::-1]
    plt.hlines(y, 0, df["diff"], color=np.where(df["diff"]>=0, "#d62728", "#1f77b4"), linewidth=2)
    plt.plot(df["diff"], y, "o", color="black")
    plt.axvline(0, color="gray", linestyle="--", linewidth=1)
    plt.yticks(y, df.index)
    plt.xlabel("Within-niche proportion − global proportion")
    plt.title(f"{condition_key} deviations in niche: {niche_of_interest}")
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.show()

In [ ]:
# Paths and keys

tma_key = "TMA_ID"

# 1) Bubble maps for IFTA and eGFR group enrichments (using saved Fisher results)
f_ifta = load_fisher_df(figure_folder_path, prefix="IFTA_group")
f_egfr = load_fisher_df(figure_folder_path, prefix="EGFR_group")

plot_bubble_enrichment(
    f_ifta,
    out_path=os.path.join(figure_folder_path, "IFTA_group_bubble.pdf"),
    title="IFTA group enrichment (Fisher, FDR)",
    min_n=20
)
plot_bubble_enrichment(
    f_egfr,
    out_path=os.path.join(figure_folder_path, "EGFR_group_bubble.pdf"),
    title="eGFR group enrichment (Fisher, FDR)",
    min_n=20
)

# 2) Stacked composition bars (cell-level)
plot_stacked_proportions(
    adata,
    latent_cluster_key=latent_cluster_key,
    condition_key="ifta_group",
    out_path=os.path.join(figure_folder_path, "ifta_group_stacked_bars.pdf"),
    min_total=50
)
plot_stacked_proportions(
    adata,
    latent_cluster_key=latent_cluster_key,
    condition_key="egfr_group",
    out_path=os.path.join(figure_folder_path, "egfr_group_stacked_bars.pdf"),
    min_total=50
)

# 3) Effect-size heatmaps (prop − global), complementary to FDR heatmaps
plot_effectsize_heatmap(
    adata,
    latent_cluster_key=latent_cluster_key,
    condition_key="ifta_group",
    out_path=os.path.join(figure_folder_path, "ifta_group_effectsize_heatmap.pdf")
)
plot_effectsize_heatmap(
    adata,
    latent_cluster_key=latent_cluster_key,
    condition_key="egfr_group",
    out_path=os.path.join(figure_folder_path, "egfr_group_effectsize_heatmap.pdf")
)

# 4) ECDFs for continuous EGFR and IFTA (sample-level)
plot_ecdf_by_niche(
    adata,
    latent_cluster_key=latent_cluster_key,
    value_key="EGFR",
    out_path=os.path.join(figure_folder_path, "EGFR_ecdf_samplelevel.pdf"),
    sample_level=True,
    tma_key=tma_key
)
plot_ecdf_by_niche(
    adata,
    latent_cluster_key=latent_cluster_key,
    value_key="IFTA_pct",
    out_path=os.path.join(figure_folder_path, "IFTA_ecdf_samplelevel.pdf"),
    sample_level=True,
    tma_key=tma_key
)

# 5) Ridgeline-style (violin + dots) for continuous values (sample-level)
plot_ridgeline_by_niche(
    adata,
    latent_cluster_key=latent_cluster_key,
    value_key="EGFR",
    out_path=os.path.join(figure_folder_path, "EGFR_ridgeline_violin.pdf"),
    tma_key=tma_key
)
plot_ridgeline_by_niche(
    adata,
    latent_cluster_key=latent_cluster_key,
    value_key="IFTA_pct",
    out_path=os.path.join(figure_folder_path, "IFTA_ridgeline_violin.pdf"),
    tma_key=tma_key
)

# 6) Lollipop deviations for a niche of interest (pick based on your results)
niche_of_interest = f_ifta.groupby("niche")["p_adj"].min().sort_values().index[0]  # best niche in IFTA
plot_lollipop_diffs(
    adata,
    latent_cluster_key=latent_cluster_key,
    condition_key="ifta_group",
    niche_of_interest=niche_of_interest,
    out_path=os.path.join(figure_folder_path, f"ifta_group_lollipop_{niche_of_interest}.pdf"),
    top_n=None  # show all groups
)

In [ ]:
# Choose a condition to visualize
condition_of_interest = adata.obs[condition_key].unique().tolist()[1]  # set explicitly

enriched_niches = sig.loc[sig["condition"] == condition_of_interest, "niche"].unique().tolist()
print("Enriched niches for", condition_of_interest, ":", enriched_niches)

# Color only enriched niches distinctly, others grey
niche_colors = {str(n): "#d3d3d3" for n in adata.obs[latent_cluster_key].cat.categories if hasattr(adata.obs[latent_cluster_key], "cat")}
# Fallback if latent_cluster_key is not categorical:
if not pd.api.types.is_categorical_dtype(adata.obs[latent_cluster_key]):
    unique_niches = adata.obs[latent_cluster_key].astype(str).unique().tolist()
    niche_colors = {n: "#d3d3d3" for n in unique_niches}
for n in enriched_niches:
    niche_colors[str(n)] = "#e41a1c"  # highlight enriched in red

# Plot per sample/TMA if available, else single plot
if "TMA_ID" in adata.obs.columns:
    for tma in adata.obs["TMA_ID"].unique():
        ad = adata[adata.obs["TMA_ID"] == tma]
        sc.pl.spatial(
            adata=ad,
            color=[latent_cluster_key],
            palette=niche_colors,
            spot_size=spot_size,
            title=f"{condition_of_interest} enriched niches (TMA {tma})",
            legend_loc=None
        )
else:
    sc.pl.spatial(
        adata=adata,
        color=[latent_cluster_key],
        palette=niche_colors,
        spot_size=spot_size,
        title=f"{condition_of_interest} enriched niches",
        legend_loc=None
    )

In [ ]:
base_path = f"{KB_XENIUM_RAW}/only_glom_roi/"

adata_roi1_path = base_path + "1734__Region_3_glom_roi.h5ad"
adata_roi2_path = base_path + "42348__Region_2_glom_roi.h5ad"
adata_roi3_path = base_path + "42348__Region_3_glom_roi.h5ad"

In [ ]:
adata_roi1 = sc.read_h5ad(adata_roi1_path)

In [ ]:
adata_roi1.obs.cell_id.head()

In [ ]:
adata.obs.cell_id.unique()

In [ ]:
adata.obs.cell_type_manual.unique()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 1) Paths
base_path = f"{KB_XENIUM_RAW}/only_glom_roi/"
adata_roi1_path = base_path + "1734__Region_3_glom_roi.h5ad"
adata_roi2_path = base_path + "42348__Region_2_glom_roi.h5ad"
adata_roi3_path = base_path + "42348__Region_3_glom_roi.h5ad"

# 2) Load ROI adatas minimally and collect in_glom per cell_id
def load_roi_map(path, id_col="cell_id", flag_col="in_glom"):
    ad = sc.read_h5ad(path)
    if id_col not in ad.obs.columns or flag_col not in ad.obs.columns:
        raise KeyError(f"{path} must contain obs['{id_col}'] and obs['{flag_col}']")
    df = ad.obs[[id_col, flag_col]].copy()
    # ensure boolean
    df[flag_col] = df[flag_col].astype(bool)
    # If duplicated cell_id, take OR (any True)
    df = df.groupby(id_col, as_index=False)[flag_col].any()
    return df

roi1 = load_roi_map(adata_roi1_path)
roi2 = load_roi_map(adata_roi2_path)
roi3 = load_roi_map(adata_roi3_path)

# 3) Merge the three into a single map: roi = roi1 OR roi2 OR roi3
roi_all = (
    roi1.rename(columns={"in_glom": "roi1"})
    .merge(roi2.rename(columns={"in_glom": "roi2"}), on="cell_id", how="outer")
    .merge(roi3.rename(columns={"in_glom": "roi3"}), on="cell_id", how="outer")
)
for c in ["roi1","roi2","roi3"]:
    roi_all[c] = roi_all[c].fillna(False).astype(bool)
roi_all["roi"] = roi_all[["roi1","roi2","roi3"]].any(axis=1)

# 4) Add roi info to your main AnnData by matching on obs["cell_id"]
adata = model.adata  # or replace with your main AnnData
if "cell_id" not in adata.obs.columns:
    raise KeyError("Main AnnData must have obs['cell_id'] to match ROI info.")

# Initialize roi as False
adata.obs["roi"] = False

# Map by cell_id
roi_map = dict(zip(roi_all["cell_id"], roi_all["roi"]))
mask_has_id = adata.obs["cell_id"].isin(roi_all["cell_id"])
adata.obs.loc[mask_has_id, "roi"] = adata.obs.loc[mask_has_id, "cell_id"].map(roi_map).fillna(False).astype(bool)

# Optional: keep also which ROI(s) matched
roi1_map = dict(zip(roi_all["cell_id"], roi_all["roi1"]))
roi2_map = dict(zip(roi_all["cell_id"], roi_all["roi2"]))
roi3_map = dict(zip(roi_all["cell_id"], roi_all["roi3"]))
for k, m in [("roi1", roi1_map), ("roi2", roi2_map), ("roi3", roi3_map)]:
    adata.obs[k] = adata.obs["cell_id"].map(m).fillna(False).astype(bool)

# 5) Compute percentage of roi=True per latent cluster

if latent_cluster_key not in adata.obs.columns:
    raise KeyError(f"{latent_cluster_key} not found in adata.obs. Ensure you ran Leiden on the latent space as in the tutorial.")

grp = adata.obs.groupby(latent_cluster_key)["roi"]
pct_in_roi = grp.mean() * 100.0  # percentage
counts = adata.obs.groupby(latent_cluster_key).size()
summary = pd.DataFrame({
    "niche": pct_in_roi.index.astype(str),
    "pct_in_roi": pct_in_roi.values,
    "n_cells": counts.values
}).sort_values("pct_in_roi", ascending=False)

summary.to_csv(f"{figure_folder_path}/roi_percentage_by_niche.csv", index=False)
display(summary.head(20))

# 6) Plot: percentage of cells in ROI per niche
plt.figure(figsize=(10, 4 + 0.2*len(pct_in_roi)))
order = summary.sort_values("pct_in_roi", ascending=False)["niche"]
sns.barplot(data=summary, x="niche", y="pct_in_roi", order=order, color="#4C78A8")
for i, (x, y, n) in enumerate(zip(order, summary.set_index("niche").loc[order, "pct_in_roi"],
                                  summary.set_index("niche").loc[order, "n_cells"])):
    plt.text(i, y + 1.0, f"n={n}", ha="center", va="bottom", fontsize=8)
plt.ylabel("% cells in ROI")
plt.xlabel("Latent niche")
plt.title("Percentage of cells in ROI per niche")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, max(5, summary["pct_in_roi"].max() * 1.15))
plt.tight_layout()
plt.savefig(f"{figure_folder_path}/roi_pct_by_niche.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# (pasted preview of three ROI tables removed for the public release: its rows carry internal core labels)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc
from pathlib import Path

# --- CONFIG ---
adata = model.adata  # your main AnnData

# Paths to the three CSVs
meta_paths = [
    f"{KB_DATA}/metadata/1734__Region_3_roi.csv",
    f"{KB_DATA}/metadata/42348__Region_2_roi.csv",
    f"{KB_DATA}/metadata/42348__Region_3_roi.csv",
]

# Output folder for figures and tables
figure_folder_path = "./figures"
Path(figure_folder_path).mkdir(parents=True, exist_ok=True)

# --- 1) Load CSVs and collect all in_* columns ---
dfs = []
all_in_cols = set()
for p in meta_paths:
    df = pd.read_csv(p)
    if "cell_id" not in df.columns:
        raise KeyError(f"{p} is missing 'cell_id' column.")
    in_cols = [c for c in df.columns if c.startswith("in_")]
    # ensure booleans
    for c in in_cols:
        # Some CSVs may store as strings "True"/"False" or 0/1
        df[c] = df[c].map({True: True, False: False, "True": True, "False": False}).fillna(df[c]).astype(bool)
    dfs.append(df[["cell_id"] + in_cols].copy())
    all_in_cols.update(in_cols)

all_in_cols = sorted(all_in_cols)
print(f"Found in_* flags: {all_in_cols}")

# --- 2) Build a combined per-cell map for each in_* flag (logical OR across files) ---
# Start with an empty DataFrame indexed by cell_id
combined = None
for i, df in enumerate(dfs):
    if combined is None:
        combined = df.set_index("cell_id")
    else:
        combined = combined.join(df.set_index("cell_id"), how="outer", rsuffix=f"__{i}")

# Now reduce columns with the same base name (since some files may not have all flags).
# Strategy: for each flag in all_in_cols, OR across all columns that equal that name or came from joins.
combined_flags = pd.DataFrame(index=combined.index.unique())
for flag in all_in_cols:
    # columns that exactly match the flag (no suffix) or have identical names from different DataFrames due to aligned join
    # Note: since we used rsuffix only when overlapping column names occurred, exact matches suffice.
    if flag in combined.columns:
        cols = [flag]
    else:
        cols = [c for c in combined.columns if c.split("__")[0] == flag or c == flag]
        if len(cols) == 0:
            # look for exact name occurrences that might have been suffixed
            cols = [c for c in combined.columns if c.endswith(flag)]
    # Extract and cast to bool, fill NaN with False
    vals = combined[cols].copy() if len(cols) > 0 else pd.DataFrame(index=combined.index)
    if vals.shape[1] == 0:
        combined_flags[flag] = False
    else:
        vals = vals.fillna(False).astype(bool)
        combined_flags[flag] = vals.any(axis=1)

combined_flags = combined_flags.reset_index().rename(columns={"index": "cell_id"})

# --- 3) Map combined flags into adata.obs via cell_id ---
if "cell_id" not in adata.obs.columns:
    raise KeyError("Main AnnData is missing obs['cell_id'].")

id_to_row = pd.Series(adata.obs_names, index=adata.obs["cell_id"]).dropna()
# Initialize all flags to False; then set True where matched
for flag in all_in_cols:
    adata.obs[flag] = False

# Map values
flag_map = combined_flags.set_index("cell_id")
matched_mask = adata.obs["cell_id"].isin(flag_map.index)
matched_ids = adata.obs.loc[matched_mask, "cell_id"]
for flag in all_in_cols:
    vals = flag_map.loc[matched_ids, flag].astype(bool).values
    adata.obs.loc[matched_mask, flag] = vals

# Any cells not present in any CSV remain False, as requested.

# --- 4) Compute percentages within each niche for every in_* flag ---
if latent_cluster_key not in adata.obs.columns:
    raise KeyError(f"{latent_cluster_key} not found in adata.obs")

results = []
for flag in all_in_cols:
    grp = adata.obs.groupby(latent_cluster_key)[flag]
    pct = grp.mean() * 100.0
    n_cells = adata.obs.groupby(latent_cluster_key).size()
    out = pd.DataFrame({
        "niche": pct.index.astype(str),
        "flag": flag,
        "pct_true": pct.values,
        "n_cells": n_cells.values,
    })
    results.append(out)

summary = pd.concat(results, axis=0)
summary.to_csv(f"{figure_folder_path}/roi_flag_percentages_by_niche.csv", index=False)
display(summary.head())

# --- 5) Plot: one heatmap of percent True per niche x flag ---
pivot = summary.pivot(index="niche", columns="flag", values="pct_true").fillna(0)
plt.figure(figsize=(max(8, 0.6*pivot.shape[1]), max(4, 0.5*pivot.shape[0])))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="viridis", cbar_kws={"label":"% cells with flag=True"})
plt.title("ROI-related flags per niche (% within niche)")
plt.xlabel("Flag")
plt.ylabel("Niche")
plt.tight_layout()
plt.savefig(f"{figure_folder_path}/roi_flags_percent_heatmap.pdf", bbox_inches="tight")
plt.show()

# --- 6) Optional: per-flag barplots (percent True per niche) ---
for flag in all_in_cols:
    sub = summary[summary["flag"] == flag].sort_values("pct_true", ascending=False)
    plt.figure(figsize=(10, 4 + 0.2*len(sub)))
    sns.barplot(data=sub, x="niche", y="pct_true", color="#4C78A8", order=sub["niche"])
    for i, (x, y, n) in enumerate(zip(sub["niche"], sub["pct_true"], sub["n_cells"])):
        plt.text(i, y + 1.0, f"n={n}", ha="center", va="bottom", fontsize=8)
    plt.ylabel("% cells with True")
    plt.xlabel("Latent niche")
    plt.title(f"{flag}: % of cells within niche that are True")
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0, max(5, sub["pct_true"].max() * 1.15))
    plt.tight_layout()
    plt.savefig(f"{figure_folder_path}/roi_flag_pct_by_niche__{flag}.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# --- CONFIG ---
adata = model.adata  # your main AnnData
latent_leiden_resolution = 0.4
latent_cluster_key = f"latent_leiden_{latent_leiden_resolution}"

meta_paths = [
    f"{KB_DATA}/processed/xenium/regions_of_interest/1734__Region_3_roi.csv",
    f"{KB_DATA}/processed/xenium/regions_of_interest/42348__Region_2_roi.csv",
    f"{KB_DATA}/processed/xenium/regions_of_interest/42348__Region_3_roi.csv",
]

figure_folder_path = "./figures_roi_flags"
Path(figure_folder_path).mkdir(parents=True, exist_ok=True)

# --- Helpers ---
def coerce_bool_series(s: pd.Series) -> pd.Series:
    def _coerce(x):
        if pd.isna(x):
            return np.nan
        if isinstance(x, bool):
            return x
        if isinstance(x, (int, np.integer)):
            return bool(x)
        if isinstance(x, (float, np.floating)):
            return bool(int(x))
        if isinstance(x, str):
            t = x.strip().lower()
            if t in {"true","t","1","yes","y"}:
                return True
            if t in {"false","f","0","no","n"}:
                return False
        return bool(x)
    return s.map(_coerce)

# --- 1) Load CSVs and gather union of in_* flags ---
dfs_raw = []
all_in_cols = set()
for p in meta_paths:
    df = pd.read_csv(p)
    # Drop unnamed index-like columns
    df = df.loc[:, ~df.columns.str.match(r"^Unnamed")]
    if "cell_id" not in df.columns:
        raise KeyError(f"{p} is missing 'cell_id' column.")
    in_cols = [c for c in df.columns if c.startswith("in_")]
    for c in in_cols:
        df[c] = coerce_bool_series(df[c])
    # Keep only cell_id + in_* columns
    dfs_raw.append(df[["cell_id"] + in_cols].copy())
    all_in_cols.update(in_cols)

all_in_cols = sorted(all_in_cols)
print(f"Detected in_* flags: {all_in_cols}")

# --- 2) Standardize frames, fill missing flags as False, OR within and across files ---
stds = []
for df in dfs_raw:
    # Ensure all flags exist; fill missing with False
    for flag in all_in_cols:
        if flag not in df.columns:
            df[flag] = False
    df_std = df[["cell_id"] + all_in_cols].copy()
    # OR within file for duplicated cell_id
    df_std = df_std.groupby("cell_id", as_index=False)[all_in_cols].any()
    stds.append(df_std)

# OR across files
combined_flags = pd.concat(stds, ignore_index=True).groupby("cell_id", as_index=False)[all_in_cols].any()

# --- 3) Map flags into adata.obs via cell_id (unmatched -> False) ---
if "cell_id" not in adata.obs.columns:
    raise KeyError("Main AnnData is missing obs['cell_id'].")

flag_map = combined_flags.set_index("cell_id")
for flag in all_in_cols:
    adata.obs[flag] = adata.obs["cell_id"].map(flag_map[flag]).fillna(False).astype(bool)

# Optional: save the combined mapping
combined_flags.to_csv(f"{figure_folder_path}/combined_in_flags_by_cell_id.csv", index=False)

# --- 4) Compute % within each niche (denominator = all cells in the niche) ---
if latent_cluster_key not in adata.obs.columns:
    raise KeyError(f"{latent_cluster_key} not found in adata.obs")

n_cells_by_niche = adata.obs.groupby(latent_cluster_key).size()

summary_list = []
for flag in all_in_cols:
    pct = adata.obs.groupby(latent_cluster_key)[flag].mean() * 100.0
    out = pd.DataFrame({
        "niche": pct.index.astype(str),
        "flag": flag,
        "pct_true": pct.values,
        "n_cells": n_cells_by_niche.values,
    })
    summary_list.append(out)

summary = pd.concat(summary_list, axis=0)
summary.to_csv(f"{figure_folder_path}/percent_true_by_niche_all_flags.csv", index=False)
display(summary.head())

# --- 5) Plot: one barplot per flag ---
import matplotlib.ticker as mtick

def plot_flag_bar(flag: str):
    sub = summary[summary["flag"] == flag].copy()
    sub = sub.sort_values("pct_true", ascending=False)
    order = sub["niche"]
    plt.figure(figsize=(10, 4 + 0.2*len(sub)))
    sns.barplot(data=sub, x="niche", y="pct_true", order=order, color="#4C78A8")
    for i, (x, y, n) in enumerate(zip(order, sub["pct_true"], sub["n_cells"])):
        plt.text(i, y + 0.8, f"n={n}", ha="center", va="bottom", fontsize=8)
    plt.ylabel("% of cells in niche")
    plt.xlabel("Latent niche")
    plt.title(f"{flag}: % of cells in each niche with {flag}=True")
    plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter())
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0, max(5, sub["pct_true"].max() * 1.15))
    plt.tight_layout()
    plt.savefig(f"{figure_folder_path}/pct_by_niche__{flag}.pdf", bbox_inches="tight")
    plt.show()

# Choose which flags to plot (set to None to plot all)
selected_flags = None  # e.g., ["in_glom", "in_fibrosis"]

flags_to_plot = [f for f in (selected_flags or all_in_cols) if f in all_in_cols]
print("Plotting flags:", flags_to_plot)
for flag in flags_to_plot:
    plot_flag_bar(flag)

# --- Optional: a single heatmap across all flags ---
pivot = summary.pivot(index="niche", columns="flag", values="pct_true").fillna(0)
plt.figure(figsize=(max(8, 0.6*pivot.shape[1]), max(4, 0.5*pivot.shape[0])))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="viridis", cbar_kws={"label":"% True within niche"})
plt.title("Percent True per niche for each in_* flag")
plt.xlabel("Flag")
plt.ylabel("Niche")
plt.tight_layout()
plt.savefig(f"{figure_folder_path}/percent_true_heatmap.pdf", bbox_inches="tight")
plt.show()

In [ ]:
adata

In [ ]:
adata.obs.cell_id.value_counts()

In [ ]:
adata.write_h5ad(f"{KB_RESULTS}/xenium/leon/xenium_with_niches.h5ad")

In [ ]:
adata.obs["latent_leiden_0.4"]

In [ ]:
import re
import pandas as pd

# 1) Helper functions
def normalize_key(x: str) -> str | None:
    """Normalize keys to a canonical form for matching."""
    if x is None:
        return None
    x = str(x).strip()
    if x == "" or x.lower() == "na":
        return None
    # Remove spaces
    x = re.sub(r"\s+", "", x)
    # If pure K-number variants like K094 or k 94 → K94
    m = re.fullmatch(r"(?i)K0*(\d+)", x)
    if m:
        return f"K{int(m.group(1))}"
    # If Apollo variants: Apollot0_A01, apollo d0 a1, apollod0A1 → APOLLOT0_A1 / APOLLOD0_A1
    m = re.fullmatch(r"(?i)(APOLLO[TD]0)_?([A-H])0*(\d+)", x)
    if m:
        base = m.group(1).upper()        # APOLLOT0 or APOLLOD0
        well = f"{m.group(2).upper()}{int(m.group(3))}"  # A1, B6, etc.
        return f"{base}_{well}"
    # General cleanup and uppercase (keeps underscores)
    return x.upper()

def derive_join_key_from_tma(tma_id: str) -> str | None:
    """Derive a normalized join key from a TMA_ID string."""
    if not isinstance(tma_id, str):
        return None
    s = tma_id

    # 1) If there's a K-number anywhere, prefer that (e.g., Utrecht1 ... K26 TIN → K26)
    m = re.search(r"\bK\s*0*(\d+)\b", s, flags=re.IGNORECASE)
    if m:
        return normalize_key(f"K{int(m.group(1))}")

    # 2) Apollo t0/d0 + well code (letter+number), e.g., "Appollo t0 F6 CTRL" → Apollot0_F6
    stage = None
    if re.search(r"\bt\s*0\b", s, flags=re.IGNORECASE):
        stage = "APOLLOT0"
    elif re.search(r"\bd\s*0\b", s, flags=re.IGNORECASE):
        stage = "APOLLOD0"

    m = re.search(r"\b([A-H])\s*0*(\d+)\b", s, flags=re.IGNORECASE)
    if stage and m:
        well = f"{m.group(1).upper()}{int(m.group(2))}"
        return f"{stage}_{well}"

    return None

# 2) Read supplemental CSV (semicolon-separated, first row is a title)
supp_path = f"{KB_PROJECT}/niche_compass_results/Supplemental.csv"
meta = pd.read_csv(supp_path, sep=";", skiprows=1, dtype=str)

# Drop stray unnamed first column if present (looks like row numbers 1,2,... in your sample)
if meta.columns[0].startswith("Unnamed") or meta.columns[0] == "":
    meta = meta.drop(columns=meta.columns[0])

# Clean whitespace in column names and values
meta.columns = [c.strip() for c in meta.columns]
meta = meta.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# 3) Build an alias → row-index map from several possible columns
possible_alias_cols = [c for c in ["sample_ID", "ck_number_rna", "ck_number_atac", "patient_ID"] if c in meta.columns]
aliases = []
for ridx, row in meta.iterrows():
    for c in possible_alias_cols:
        alias = normalize_key(row[c])
        if alias is not None:
            aliases.append({"alias": alias, "meta_idx": ridx})

aliases_df = pd.DataFrame(aliases)
# Keep first occurrence if duplicates; warn if duplicates exist
dups = aliases_df["alias"].duplicated(keep=False)
if dups.any():
    dup_aliases = aliases_df.loc[dups, "alias"].unique().tolist()
    print(f"Warning: duplicate aliases in metadata for {len(dup_aliases)} keys. Using first occurrence:\n{dup_aliases[:20]}{' ...' if len(dup_aliases)>20 else ''}")

aliases_df = aliases_df.drop_duplicates(subset="alias", keep="first")
alias_to_metaidx = aliases_df.set_index("alias")["meta_idx"]

# 4) Derive join keys from adata.obs['TMA_ID'] and map to metadata rows
# Assumes you already have 'adata' loaded and adata.obs['TMA_ID'] present
def attach_supplemental_to_adata(adata, meta_df, alias_to_metaidx, prefix="supp_"):
    if "TMA_ID" not in adata.obs.columns:
        raise KeyError("adata.obs must contain a 'TMA_ID' column.")

    join_keys = adata.obs["TMA_ID"].apply(derive_join_key_from_tma)
    # Map to metadata row indices
    meta_idx_series = join_keys.map(alias_to_metaidx)

    # Build a DataFrame aligned to adata.obs.index with supplemental columns
    supp_cols = [c for c in meta_df.columns]  # keep all columns
    out = pd.DataFrame(index=adata.obs.index, columns=[prefix + c for c in supp_cols])

    mask = meta_idx_series.notna()
    if mask.any():
        # meta_df.loc[indices] will return rows in the order of indices
        matched_meta = meta_df.loc[meta_idx_series[mask].astype(int)].set_index(adata.obs.index[mask])
        out.loc[mask, :] = matched_meta.rename(columns=lambda c: prefix + c)

    # Attach to adata.obs
    adata.obs = pd.concat([adata.obs, out], axis=1)

    # Simple report
    n_matched = int(mask.sum())
    n_total = adata.n_obs
    print(f"Matched {n_matched}/{n_total} observations "
          f"({n_matched/n_total:.1%}). Unmatched: {n_total - n_matched}.")

    # Optionally, inspect a few unmatched TMA_IDs
    unmatched_examples = adata.obs.loc[~mask, "TMA_ID"].dropna().unique().tolist()[:20]
    if len(unmatched_examples) > 0:
        print("Examples of unmatched TMA_IDs (first 20):")
        for u in unmatched_examples:
            print(" -", u)

# 5) Run the attachment
# from anndata import AnnData  # if needed
# import scanpy as sc          # if using Scanpy
attach_supplemental_to_adata(adata, meta, alias_to_metaidx, prefix="supp_")